In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub

In [7]:
from pathlib import Path
import shutil

MATRIX_DIR = Path("/kaggle/working/multiome_sparse")
MATRIX_DIR.mkdir(parents=True, exist_ok=True)

matrix_names = [
    "train_multi_inputs_matrix.npz",
    "test_multi_inputs_matrix.npz",
    "train_multi_targets_matrix.npz",
]

# 自动寻找同时包含三个矩阵的 Input 目录
candidates = sorted({
    p.parent
    for p in Path("/kaggle/input").rglob(matrix_names[0])
    if all((p.parent / name).is_file() for name in matrix_names)
})

if len(candidates) != 1:
    raise RuntimeError(
        f"找到 {len(candidates)} 个匹配目录：{candidates}\n"
        "请检查 Multiomics_matrix 是否已添加，且文件名保持原样。"
    )

dataset_dir = candidates[0]
print("Dataset 目录：", dataset_dir)

for name in matrix_names:
    source = dataset_dir / name
    destination = MATRIX_DIR / name

    if destination.exists():
        if destination.stat().st_size != source.stat().st_size:
            raise RuntimeError(f"已有文件大小不一致，请检查：{destination}")
        print(f"已存在，跳过：{name}")
        continue

    print(f"正在复制：{name}", flush=True)

    # 先写入临时文件，复制完成后才使用正式文件名
    temporary = destination.with_name(destination.name + ".copying")
    shutil.copy2(source, temporary)
    temporary.replace(destination)

    print(f"复制完成：{name}", flush=True)

print("\n三个矩阵已就位：", MATRIX_DIR)

Dataset 目录： /kaggle/input/datasets/huangyufeid/multiomics-matrix
正在复制：train_multi_inputs_matrix.npz
复制完成：train_multi_inputs_matrix.npz
正在复制：test_multi_inputs_matrix.npz
复制完成：test_multi_inputs_matrix.npz
正在复制：train_multi_targets_matrix.npz
复制完成：train_multi_targets_matrix.npz

三个矩阵已就位： /kaggle/working/multiome_sparse


In [34]:
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp

# 你的 Kaggle 比赛数据实际路径
DATA_DIR = Path(
    "/kaggle/input/competitions/open-problems-multimodal"
)

# 后续中间文件保存位置
WORK_DIR = Path("/kaggle/working/multiome_sparse")
WORK_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "train_inputs": DATA_DIR / "train_multi_inputs.h5",
    "train_targets": DATA_DIR / "train_multi_targets.h5",
    "test_inputs": DATA_DIR / "test_multi_inputs.h5",
    "metadata": DATA_DIR / "metadata.csv",
    "evaluation_ids": DATA_DIR / "evaluation_ids.csv",
    "sample_submission": DATA_DIR / "sample_submission.csv",
}

for name, path in PATHS.items():
    print(f"{name:18s} | exists={path.exists()} | {path}")

train_inputs       | exists=True | /kaggle/input/competitions/open-problems-multimodal/train_multi_inputs.h5
train_targets      | exists=True | /kaggle/input/competitions/open-problems-multimodal/train_multi_targets.h5
test_inputs        | exists=True | /kaggle/input/competitions/open-problems-multimodal/test_multi_inputs.h5
metadata           | exists=True | /kaggle/input/competitions/open-problems-multimodal/metadata.csv
evaluation_ids     | exists=True | /kaggle/input/competitions/open-problems-multimodal/evaluation_ids.csv
sample_submission  | exists=True | /kaggle/input/competitions/open-problems-multimodal/sample_submission.csv


In [35]:
def inspect_hdf(path, nrows=5):
    with pd.HDFStore(path, mode="r") as store:
        keys = store.keys()

        print(f"\nFile: {path.name}")
        print("HDF keys:", keys)

        key = keys[0]
        storer = store.get_storer(key)

        print("Stored shape:", getattr(storer, "shape", None))
        print("Stored nrows:", getattr(storer, "nrows", None))

    sample = pd.read_hdf(path, start=0, stop=nrows)

    print("Sample shape:", sample.shape)
    print("Index examples:", sample.index[:3].tolist())
    print("Column examples:", sample.columns[:5].tolist())
    print("Dtypes:")
    print(sample.dtypes.value_counts())

    return sample


x_train_sample = inspect_hdf(PATHS["train_inputs"])
y_train_sample = inspect_hdf(PATHS["train_targets"])
x_test_sample  = inspect_hdf(PATHS["test_inputs"])


File: train_multi_inputs.h5
HDF keys: ['/train_multi_inputs']
Stored shape: [np.int64(105942), np.int64(228942)]
Stored nrows: None
Sample shape: (5, 228942)
Index examples: ['56390cf1b95e', 'fc0c60183c33', '9b4a87e22ad0']
Column examples: ['GL000194.1:114519-115365', 'GL000194.1:55758-56597', 'GL000194.1:58217-58957', 'GL000194.1:59535-60431', 'GL000195.1:119766-120427']
Dtypes:
float32    228942
Name: count, dtype: int64

File: train_multi_targets.h5
HDF keys: ['/train_multi_targets']
Stored shape: [np.int64(105942), np.int64(23418)]
Stored nrows: None
Sample shape: (5, 23418)
Index examples: ['56390cf1b95e', 'fc0c60183c33', '9b4a87e22ad0']
Column examples: ['ENSG00000121410', 'ENSG00000268895', 'ENSG00000175899', 'ENSG00000245105', 'ENSG00000166535']
Dtypes:
float32    23418
Name: count, dtype: int64

File: test_multi_inputs.h5
HDF keys: ['/test_multi_inputs']
Stored shape: [np.int64(55935), np.int64(228942)]
Stored nrows: None
Sample shape: (5, 228942)
Index examples: ['458c2ae2c9

In [36]:
import os
import gc
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp

from tqdm.auto import tqdm


def hdf_to_csr(
    h5_path,
    output_prefix,
    chunk_rows=512
):
    """
    分块读取 pandas HDF5，并转换为 CSR 稀疏矩阵。

    输出三个文件：
    1. *_matrix.npz：稀疏矩阵
    2. *_rows.npy：cell_id
    3. *_columns.npy：feature/gene ID
    """

    matrix_path = WORK_DIR / f"{output_prefix}_matrix.npz"
    rows_path = WORK_DIR / f"{output_prefix}_rows.npy"
    columns_path = WORK_DIR / f"{output_prefix}_columns.npy"

    # 如果已经完整转换过，则跳过
    if (
        matrix_path.exists()
        and rows_path.exists()
        and columns_path.exists()
    ):
        print(f"{output_prefix}: files already exist, skipping.")
        return

    with pd.HDFStore(h5_path, mode="r") as store:
        key = store.keys()[0]
        shape = store.get_storer(key).shape

    n_rows = int(shape[0])
    n_columns = int(shape[1])

    print(f"\nConverting: {h5_path.name}")
    print(f"Shape: ({n_rows:,}, {n_columns:,})")
    print(f"Chunk size: {chunk_rows}")

    sparse_blocks = []
    row_ids = []
    column_ids = None

    start_time = time.time()

    for start in tqdm(
        range(0, n_rows, chunk_rows),
        desc=output_prefix
    ):
        stop = min(start + chunk_rows, n_rows)

        chunk = pd.read_hdf(
            h5_path,
            key=key,
            start=start,
            stop=stop
        )

        # 第一块记录完整列名
        if column_ids is None:
            column_ids = chunk.columns.astype(str).to_numpy()

        row_ids.extend(chunk.index.astype(str).tolist())

        values = chunk.to_numpy(
            dtype=np.float32,
            copy=False
        )

        sparse_chunk = sp.csr_matrix(values)
        sparse_chunk.eliminate_zeros()

        sparse_blocks.append(sparse_chunk)

        del chunk, values, sparse_chunk
        gc.collect()

    print("Combining sparse blocks...")

    matrix = sp.vstack(
        sparse_blocks,
        format="csr",
        dtype=np.float32
    )

    del sparse_blocks
    gc.collect()

    row_ids = np.asarray(row_ids, dtype=str)
    column_ids = np.asarray(column_ids, dtype=str)

    assert matrix.shape == (n_rows, n_columns)
    assert len(row_ids) == n_rows
    assert len(column_ids) == n_columns

    density = matrix.nnz / (matrix.shape[0] * matrix.shape[1])

    print("Final shape:", matrix.shape)
    print(f"Non-zero values: {matrix.nnz:,}")
    print(f"Density: {density:.6%}")

    # 使用临时文件，防止运行中断后留下损坏文件
    temp_matrix = WORK_DIR / f"{output_prefix}_matrix.tmp.npz"
    temp_rows = WORK_DIR / f"{output_prefix}_rows.tmp.npy"
    temp_columns = WORK_DIR / f"{output_prefix}_columns.tmp.npy"

    print("Saving sparse matrix...")

    sp.save_npz(
        temp_matrix,
        matrix,
        compressed=True
    )

    np.save(
        temp_rows,
        row_ids,
        allow_pickle=False
    )

    np.save(
        temp_columns,
        column_ids,
        allow_pickle=False
    )

    os.replace(temp_matrix, matrix_path)
    os.replace(temp_rows, rows_path)
    os.replace(temp_columns, columns_path)

    elapsed = (time.time() - start_time) / 60

    print(f"Saved to: {matrix_path}")
    print(f"Elapsed time: {elapsed:.1f} minutes")

    del matrix, row_ids, column_ids
    gc.collect()

以下3个block用于转换.npz，不用运行：

In [ ]:
hdf_to_csr(
    h5_path=PATHS["train_inputs"],
    output_prefix="train_multi_inputs",
    chunk_rows=512
)

In [ ]:
hdf_to_csr(
    h5_path=PATHS["train_targets"],
    output_prefix="train_multi_targets",
    chunk_rows=2048
)

In [ ]:
hdf_to_csr(
    h5_path=PATHS["test_inputs"],
    output_prefix="test_multi_inputs",
    chunk_rows=512
)

In [37]:
def inspect_sparse(prefix):
    matrix_path = WORK_DIR / f"{prefix}_matrix.npz"

    matrix = sp.load_npz(matrix_path)

    density = matrix.nnz / (
        matrix.shape[0] * matrix.shape[1]
    )

    print(
        f"{prefix:22s} | "
        f"shape={matrix.shape} | "
        f"nnz={matrix.nnz:,} | "
        f"density={density:.6%}"
    )

    del matrix
    gc.collect()


inspect_sparse("train_multi_inputs")
inspect_sparse("train_multi_targets")
inspect_sparse("test_multi_inputs")

train_multi_inputs     | shape=(105942, 228942) | nnz=607,301,546 | density=2.503864%
train_multi_targets    | shape=(105942, 23418) | nnz=407,024,875 | density=16.406010%
test_multi_inputs      | shape=(55935, 228942) | nnz=353,572,710 | density=2.761020%


In [24]:
from pathlib import Path
import os
import time
import numpy as np
import pandas as pd

WORK_DIR = Path("/kaggle/working/multiome_sparse")
WORK_DIR.mkdir(parents=True, exist_ok=True)


def recover_hdf_labels(h5_path, output_prefix):
    """只恢复 rows.npy 和 columns.npy，不修改已有 matrix.npz。"""
    h5_path = Path(h5_path)

    matrix_path = WORK_DIR / f"{output_prefix}_matrix.npz"
    rows_path = WORK_DIR / f"{output_prefix}_rows.npy"
    columns_path = WORK_DIR / f"{output_prefix}_columns.npy"

    if not matrix_path.is_file():
        raise FileNotFoundError(f"请先恢复已有矩阵：{matrix_path}")

    started = time.perf_counter()
    print(f"\n恢复标签：{output_prefix}", flush=True)
    print("原始文件：", h5_path)

    # 只读取 NPZ 中很小的 shape 数组，不加载矩阵数值
    with np.load(matrix_path, allow_pickle=False) as archive:
        matrix_shape = tuple(int(x) for x in archive["shape"])

    with pd.HDFStore(h5_path, mode="r") as store:
        # 与原 hdf_to_csr 使用相同的 key 选择方式
        keys = store.keys()
        if not keys:
            raise RuntimeError(f"HDF5 中没有 pandas 数据：{h5_path}")

        key = keys[0]
        storer = store.get_storer(key)

        if storer.is_table:
            # table 格式：单独读取 index；只读一行获取列名
            row_index = store.select_column(key, "index")
            first_row = store.select(key, start=0, stop=1)
            column_index = first_row.columns
        else:
            # fixed 格式：直接读取两个索引，不读取数值块
            # read_index 是 pandas storer 的内部方法
            row_index = storer.read_index("axis1")
            column_index = storer.read_index("axis0")

        # 与原函数的字符串转换方式一致，不排序、不重排
        row_ids = np.asarray(row_index.astype(str), dtype=str)
        column_ids = np.asarray(column_index.astype(str), dtype=str)

    labels_shape = (len(row_ids), len(column_ids))
    if labels_shape != matrix_shape:
        raise RuntimeError(
            f"标签尺寸 {labels_shape} 与矩阵尺寸 {matrix_shape} 不一致；"
            "请检查是否选择了原来使用的 HDF5。"
        )

    outputs = [
        (rows_path, row_ids),
        (columns_path, column_ids),
    ]

    # 已有标签必须与原始 HDF5 一致，避免覆盖不同来源的文件
    for path, values in outputs:
        if path.exists():
            existing = np.load(path, allow_pickle=False)
            if not np.array_equal(existing, values):
                raise RuntimeError(f"已有标签与原始 HDF5 不一致：{path}")

    for path, values in outputs:
        if path.exists():
            print("已存在且一致，保留：", path.name)
            continue

        temporary = path.with_name(path.stem + ".tmp.npy")
        np.save(temporary, values, allow_pickle=False)
        os.replace(temporary, path)
        print("已生成：", path.name)

    print("矩阵尺寸：", matrix_shape)
    print(f"耗时：{time.perf_counter() - started:.1f} 秒")


# 自动查找三个原始 HDF5 文件
prefixes = [
    "train_multi_inputs",
    "train_multi_targets",
    "test_multi_inputs",
]

# 先确认全部路径，避免选错同名文件
h5_paths = {}

for prefix in prefixes:
    matches = sorted(
        Path("/kaggle/input").rglob(f"{prefix}.h5")
    )
    if len(matches) != 1:
        raise RuntimeError(
            f"{prefix}.h5 找到 {len(matches)} 个匹配文件：{matches}\n"
            "请确认原始竞赛数据已添加，且使用原来转换的文件。"
        )
    h5_paths[prefix] = matches[0]

for prefix in prefixes:
    recover_hdf_labels(h5_paths[prefix], prefix)

print("\n六个标签文件恢复完成，可以运行原来的 alignment checks。")


恢复标签：train_multi_inputs
原始文件： /kaggle/input/competitions/open-problems-multimodal/train_multi_inputs.h5
已生成： train_multi_inputs_rows.npy
已生成： train_multi_inputs_columns.npy
矩阵尺寸： (105942, 228942)
耗时：0.5 秒

恢复标签：train_multi_targets
原始文件： /kaggle/input/competitions/open-problems-multimodal/train_multi_targets.h5
已生成： train_multi_targets_rows.npy
已生成： train_multi_targets_columns.npy
矩阵尺寸： (105942, 23418)
耗时：0.1 秒

恢复标签：test_multi_inputs
原始文件： /kaggle/input/competitions/open-problems-multimodal/test_multi_inputs.h5
已生成： test_multi_inputs_rows.npy
已生成： test_multi_inputs_columns.npy
矩阵尺寸： (55935, 228942)
耗时：0.5 秒

六个标签文件恢复完成，可以运行原来的 alignment checks。


In [39]:
train_input_cells = np.load(
    WORK_DIR / "train_multi_inputs_rows.npy",
    allow_pickle=False
)

train_target_cells = np.load(
    WORK_DIR / "train_multi_targets_rows.npy",
    allow_pickle=False
)

test_input_cells = np.load(
    WORK_DIR / "test_multi_inputs_rows.npy",
    allow_pickle=False
)

train_atac_features = np.load(
    WORK_DIR / "train_multi_inputs_columns.npy",
    allow_pickle=False
)

test_atac_features = np.load(
    WORK_DIR / "test_multi_inputs_columns.npy",
    allow_pickle=False
)

target_genes = np.load(
    WORK_DIR / "train_multi_targets_columns.npy",
    allow_pickle=False
)

assert np.array_equal(
    train_input_cells,
    train_target_cells
), "Train input 与 target 的 cell_id 不一致"

assert np.array_equal(
    train_atac_features,
    test_atac_features
), "Train 与 test 的 ATAC feature 顺序不一致"

print("All alignment checks passed.")
print("Train cells:", len(train_input_cells))
print("Test cells:", len(test_input_cells))
print("ATAC features:", len(train_atac_features))
print("Target genes:", len(target_genes))

All alignment checks passed.
Train cells: 105942
Test cells: 55935
ATAC features: 228942
Target genes: 23418


In [40]:
import time
import numpy as np
import pandas as pd

started = time.perf_counter()

metadata = pd.read_csv(
    PATHS["metadata"],
    dtype={
        "cell_id": str,
        "donor": str,
        "technology": str,
        "cell_type": str,
    }
)

print("Full metadata shape:", metadata.shape)
print(metadata.head())
print("\nColumns:", metadata.columns.tolist())
print(
    f"读取 CSV：{time.perf_counter() - started:.2f} 秒",
    flush=True
)

started = time.perf_counter()

assert metadata["cell_id"].is_unique, \
    "metadata 中存在重复 cell_id"

metadata_indexed = metadata.set_index("cell_id")

print("Cell array dtype:", train_input_cells.dtype)
print("Metadata index dtype:", metadata_indexed.index.dtype)
print(
    f"建立索引：{time.perf_counter() - started:.2f} 秒",
    flush=True
)

started = time.perf_counter()

# 返回位置的顺序与传入的 cell_id 顺序一致
train_positions = metadata_indexed.index.get_indexer(
    train_input_cells
)
test_positions = metadata_indexed.index.get_indexer(
    test_input_cells
)

missing_train = np.unique(
    train_input_cells[train_positions < 0]
)
missing_test = np.unique(
    test_input_cells[test_positions < 0]
)

print("Missing train cells:", len(missing_train))
print("Missing test cells:", len(missing_test))
print(
    f"匹配 cell_id：{time.perf_counter() - started:.2f} 秒",
    flush=True
)

# 必须先确认没有 -1，才能使用 iloc
assert len(missing_train) == 0, \
    f"训练集缺失 ID，例如：{missing_train[:5]}"
assert len(missing_test) == 0, \
    f"测试集缺失 ID，例如：{missing_test[:5]}"

started = time.perf_counter()

train_meta = metadata_indexed.iloc[
    train_positions
].reset_index()

test_meta = metadata_indexed.iloc[
    test_positions
].reset_index()

train_meta["split"] = "train"
test_meta["split"] = "test"

# 核对 metadata 与稀疏矩阵的细胞顺序
assert np.array_equal(
    train_meta["cell_id"].to_numpy(),
    train_input_cells
), "Train metadata 顺序不一致"

assert np.array_equal(
    test_meta["cell_id"].to_numpy(),
    test_input_cells
), "Test metadata 顺序不一致"

print("\nTrain metadata:", train_meta.shape)
print("Test metadata:", test_meta.shape)
print(
    f"排列 metadata：{time.perf_counter() - started:.2f} 秒",
    flush=True
)
print("All metadata alignment checks passed.")

Full metadata shape: (281528, 5)
        cell_id  day  donor cell_type technology
0  c2150f55becb    2  27678       HSC    citeseq
1  65b7edf8a4da    2  27678       HSC    citeseq
2  c1b26cb1057b    2  27678      EryP    citeseq
3  917168fa6f83    2  27678      NeuP    citeseq
4  2b29feeca86d    2  27678      EryP    citeseq

Columns: ['cell_id', 'day', 'donor', 'cell_type', 'technology']
读取 CSV：0.31 秒
Cell array dtype: <U12
Metadata index dtype: object
建立索引：0.10 秒
Missing train cells: 0
Missing test cells: 0
匹配 cell_id：0.12 秒

Train metadata: (105942, 6)
Test metadata: (55935, 6)
排列 metadata：0.06 秒
All metadata alignment checks passed.


In [41]:
print("Train technologies:")
print(train_meta["technology"].value_counts(dropna=False))

print("\nTest technologies:")
print(test_meta["technology"].value_counts(dropna=False))

assert train_meta["cell_id"].tolist() == train_input_cells.tolist()
assert test_meta["cell_id"].tolist() == test_input_cells.tolist()

print("\nMetadata order matches sparse matrices.")

Train technologies:
technology
multiome    105942
Name: count, dtype: int64

Test technologies:
technology
multiome    55935
Name: count, dtype: int64

Metadata order matches sparse matrices.


Day and donor distribution

In [42]:
multiome_meta = pd.concat(
    [train_meta, test_meta],
    ignore_index=True
)

print("Train/test × day:")
display(
    pd.crosstab(
        multiome_meta["split"],
        multiome_meta["day"]
    )
)

print("\nTrain/test × donor:")
display(
    pd.crosstab(
        multiome_meta["split"],
        multiome_meta["donor"]
    )
)

print("\nDonor × day × split:")
donor_day_table = (
    multiome_meta
    .groupby(
        ["donor", "day", "split"],
        observed=True
    )
    .size()
    .unstack("split", fill_value=0)
    .sort_index()
)

display(donor_day_table)

Train/test × day:


day,2,3,4,7,10
split,,,,,
test,8921,7950,0,7291,31773
train,23911,28815,31134,22082,0



Train/test × donor:


donor,13176,27678,31800,32606
split,,,,
test,8421,32952,7144,7418
train,35396,0,36845,33701



Donor × day × split:


split      test  train
donor day             
13176 2       0   9558
      3       0   8965
      4       0   9407
      7       0   7466
      10   8421      0
27678 2    8921      0
      3    7950      0
      7    7291      0
      10   8790      0
31800 2       0   7089
      3       0  11306
      4       0  10794
      7       0   7656
      10   7144      0
32606 2       0   7264
      3       0   8544
      4       0  10933
      7       0   6960
      10   7418      0

cell type information

In [43]:
print("Train cell types:")
print(
    train_meta["cell_type"]
    .value_counts(dropna=False)
)

print("\nTest cell types:")
print(
    test_meta["cell_type"]
    .value_counts(dropna=False)
)

Train cell types:
cell_type
HSC     34661
NeuP    21561
EryP    17135
MasP    16737
MkP     13221
MoP      2061
BP        566
Name: count, dtype: int64

Test cell types:
cell_type
hidden    55935
Name: count, dtype: int64


In [44]:
train_meta["donor_day"] = (
    train_meta["donor"].astype(str)
    + "_day_"
    + train_meta["day"].astype(str)
)

test_meta["donor_day"] = (
    test_meta["donor"].astype(str)
    + "_day_"
    + test_meta["day"].astype(str)
)

print("Training donor-day groups:")
print(
    train_meta["donor_day"]
    .value_counts()
    .sort_index()
)

print("\nTest donor-day groups:")
print(
    test_meta["donor_day"]
    .value_counts()
    .sort_index()
)

Training donor-day groups:
donor_day
13176_day_2     9558
13176_day_3     8965
13176_day_4     9407
13176_day_7     7466
31800_day_2     7089
31800_day_3    11306
31800_day_4    10794
31800_day_7     7656
32606_day_2     7264
32606_day_3     8544
32606_day_4    10933
32606_day_7     6960
Name: count, dtype: int64

Test donor-day groups:
donor_day
13176_day_10    8421
27678_day_10    8790
27678_day_2     8921
27678_day_3     7950
27678_day_7     7291
31800_day_10    7144
32606_day_10    7418
Name: count, dtype: int64


In [45]:
train_meta.to_csv(
    WORK_DIR / "train_multi_metadata.csv",
    index=False
)

test_meta.to_csv(
    WORK_DIR / "test_multi_metadata.csv",
    index=False
)

print("Aligned metadata saved.")

Aligned metadata saved.


## Tune parameters

In [46]:
VALID_DONOR = "32606"
VALID_DAY = 7

donor_values = train_meta["donor"].astype(str)
day_values = train_meta["day"].astype(int)

# 训练：既不能见过验证 donor，也不能见过验证 day
fit_mask = (
    (donor_values != VALID_DONOR)
    & (day_values != VALID_DAY)
).to_numpy()

# 验证：未见 donor + 未来 day 的交集
valid_mask = (
    (donor_values == VALID_DONOR)
    & (day_values == VALID_DAY)
).to_numpy()

fit_idx = np.flatnonzero(fit_mask)
valid_idx = np.flatnonzero(valid_mask)

print("Strict training cells:", len(fit_idx))
print("Strict validation cells:", len(valid_idx))
print("Unused cells:", len(train_meta) - len(fit_idx) - len(valid_idx))

print("\nTraining donor × day:")
display(
    pd.crosstab(
        train_meta.loc[fit_idx, "donor"],
        train_meta.loc[fit_idx, "day"]
    )
)

print("\nValidation donor × day:")
display(
    pd.crosstab(
        train_meta.loc[valid_idx, "donor"],
        train_meta.loc[valid_idx, "day"]
    )
)

Strict training cells: 57119
Strict validation cells: 6960
Unused cells: 41863

Training donor × day:


day,2,3,4
donor,,,
13176,9558,8965,9407
31800,7089,11306,10794



Validation donor × day:


day,7
donor,
32606,6960


In [47]:
import scipy.sparse as sp
import gc

X_train = sp.load_npz(
    WORK_DIR / "train_multi_inputs_matrix.npz"
).astype(np.float32)

X_test = sp.load_npz(
    WORK_DIR / "test_multi_inputs_matrix.npz"
).astype(np.float32)

Y_train = sp.load_npz(
    WORK_DIR / "train_multi_targets_matrix.npz"
).astype(np.float32)

print("X_train:", X_train.shape, X_train.dtype)
print("X_test: ", X_test.shape, X_test.dtype)
print("Y_train:", Y_train.shape, Y_train.dtype)

print("\nATAC non-zero value range:")
print("min:", X_train.data.min())
print("max:", X_train.data.max())
print("mean:", X_train.data.mean())

X_train: (105942, 228942) float32
X_test:  (55935, 228942) float32
Y_train: (105942, 23418) float32

ATAC non-zero value range:
min: 0.032380838
max: 18.601044
mean: 1.9907693


128 SVD on ATAC-seq data

In [ ]:
from sklearn.decomposition import TruncatedSVD
import joblib
import time

N_COMPONENTS_X = 128

print("Combining train and test ATAC matrices...")

# 只使用输入数据，不涉及测试标签，因此可以共同学习降维空间
X_all = sp.vstack(
    [X_train, X_test],
    format="csr"
)

print("Combined shape:", X_all.shape)

input_svd = TruncatedSVD(
    n_components=N_COMPONENTS_X,
    algorithm="randomized",
    n_iter=5,
    random_state=42
)

start_time = time.time()

X_all_svd = input_svd.fit_transform(X_all).astype(np.float32)

elapsed = (time.time() - start_time) / 60

print("SVD output shape:", X_all_svd.shape)
print(f"Elapsed: {elapsed:.1f} minutes")
print(
    "Explained variance ratio:",
    input_svd.explained_variance_ratio_.sum()
)

X_train_svd = X_all_svd[:len(train_input_cells)]
X_test_svd = X_all_svd[len(train_input_cells):]

assert X_train_svd.shape == (105942, 128)
assert X_test_svd.shape == (55935, 128)

np.save(
    WORK_DIR / "X_train_multi_svd128.npy",
    X_train_svd,
    allow_pickle=False
)

np.save(
    WORK_DIR / "X_test_multi_svd128.npy",
    X_test_svd,
    allow_pickle=False
)

joblib.dump(
    input_svd,
    WORK_DIR / "input_multi_svd128.joblib"
)

del X_all, X_all_svd, X_train, X_test
gc.collect()

print("Input SVD artifacts saved.")

In [ ]:
N_COMPONENTS_Y = 128

target_svd_valid = TruncatedSVD(
    n_components=N_COMPONENTS_Y,
    algorithm="randomized",
    n_iter=7,
    random_state=42
)

start_time = time.time()

Y_fit_svd = target_svd_valid.fit_transform(
    Y_train[fit_idx]
).astype(np.float32)

elapsed = (time.time() - start_time) / 60

print("Y-fit latent shape:", Y_fit_svd.shape)
print(f"Elapsed: {elapsed:.1f} minutes")
print(
    "Target explained variance ratio:",
    target_svd_valid.explained_variance_ratio_.sum()
)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

ridge_model = make_pipeline(
    StandardScaler(),
    Ridge(
        alpha=10.0,
        fit_intercept=True
    )
)

ridge_model.fit(
    X_train_svd[fit_idx],
    Y_fit_svd
)

print("Ridge training completed.")

In [ ]:
def rowwise_pearson(y_true, y_pred):
    y_true = y_true.astype(np.float32, copy=False)
    y_pred = y_pred.astype(np.float32, copy=False)

    y_true_centered = (
        y_true
        - y_true.mean(axis=1, keepdims=True)
    )

    y_pred_centered = (
        y_pred
        - y_pred.mean(axis=1, keepdims=True)
    )

    numerator = np.sum(
        y_true_centered * y_pred_centered,
        axis=1
    )

    denominator = np.sqrt(
        np.sum(y_true_centered ** 2, axis=1)
        * np.sum(y_pred_centered ** 2, axis=1)
    )

    return np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator),
        where=denominator > 0
    )


def evaluate_latent_predictions(
    y_true_sparse,
    z_pred,
    target_svd,
    batch_size=256
):
    raw_correlations = []
    clipped_correlations = []

    for start in range(0, y_true_sparse.shape[0], batch_size):
        stop = min(
            start + batch_size,
            y_true_sparse.shape[0]
        )

        y_true_batch = (
            y_true_sparse[start:stop]
            .toarray()
            .astype(np.float32)
        )

        y_pred_batch = (
            target_svd
            .inverse_transform(z_pred[start:stop])
            .astype(np.float32)
        )

        raw_correlations.append(
            rowwise_pearson(
                y_true_batch,
                y_pred_batch
            )
        )

        # GEX 不应为负数，同时检查截断后是否更好
        np.maximum(
            y_pred_batch,
            0,
            out=y_pred_batch
        )

        clipped_correlations.append(
            rowwise_pearson(
                y_true_batch,
                y_pred_batch
            )
        )

        del y_true_batch, y_pred_batch
        gc.collect()

    raw_score = np.concatenate(
        raw_correlations
    ).mean()

    clipped_score = np.concatenate(
        clipped_correlations
    ).mean()

    return raw_score, clipped_score

In [ ]:
Y_valid = Y_train[valid_idx]

Z_valid_oracle = (
    target_svd_valid
    .transform(Y_valid)
    .astype(np.float32)
)

oracle_raw, oracle_clipped = evaluate_latent_predictions(
    Y_valid,
    Z_valid_oracle,
    target_svd_valid
)

print("Target-SVD reconstruction score")
print("Raw:    ", oracle_raw)
print("Clipped:", oracle_clipped)

In [ ]:
Z_valid_pred = ridge_model.predict(
    X_train_svd[valid_idx]
).astype(np.float32)

ridge_raw, ridge_clipped = evaluate_latent_predictions(
    Y_valid,
    Z_valid_pred,
    target_svd_valid
)

print("Strict validation Ridge score")
print("Raw:    ", ridge_raw)
print("Clipped:", ridge_clipped)

In [ ]:
Y_valid = Y_train[valid_idx]

Z_valid_oracle = (
    target_svd_valid
    .transform(Y_valid)
    .astype(np.float32)
)

oracle_raw, oracle_clipped = evaluate_latent_predictions(
    Y_valid,
    Z_valid_oracle,
    target_svd_valid,
    batch_size=256
)

print("Target-SVD reconstruction ceiling")
print(f"Raw:     {oracle_raw:.6f}")
print(f"Clipped: {oracle_clipped:.6f}")

print("\nRidge model")
print(f"Raw:     {ridge_raw:.6f}")
print(f"Clipped: {ridge_clipped:.6f}")

print("\nGap between SVD ceiling and Ridge:")
print(f"Raw gap: {oracle_raw - ridge_raw:.6f}")

In [ ]:
def evaluate_fixed_prediction(
    y_true_sparse,
    prediction_vector,
    batch_size=256
):
    correlations = []

    for start in range(
        0,
        y_true_sparse.shape[0],
        batch_size
    ):
        stop = min(
            start + batch_size,
            y_true_sparse.shape[0]
        )

        y_true_batch = (
            y_true_sparse[start:stop]
            .toarray()
            .astype(np.float32)
        )

        y_pred_batch = np.broadcast_to(
            prediction_vector,
            y_true_batch.shape
        )

        correlations.append(
            rowwise_pearson(
                y_true_batch,
                y_pred_batch
            )
        )

    return np.concatenate(correlations).mean()


gene_mean = np.asarray(
    Y_train[fit_idx].mean(axis=0)
).ravel().astype(np.float32)

mean_baseline_score = evaluate_fixed_prediction(
    Y_valid,
    gene_mean
)

print(
    "Gene-mean baseline:",
    f"{mean_baseline_score:.6f}"
)

print(
    "Ridge improvement:",
    f"{ridge_clipped - mean_baseline_score:.6f}"
)

| 指标 | 分数 |
|---|---:|
| 所有细胞预测相同 gene mean | 0.5600 |
| Ridge | 0.5944 |
| 128维 Target SVD 理论上限 | 0.6298 |

128 Target SVD is the main drawback.

2 changes:      
- do L2 normalization on every cell target
- increase SVD from 128 to 256

In [48]:
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

import time
import gc


def run_target_experiment(
    n_components,
    row_normalize=True,
    ridge_alpha=10.0
):
    print("=" * 60)
    print(
        f"Target components: {n_components}, "
        f"row normalization: {row_normalize}"
    )

    Y_fit_original = Y_train[fit_idx]
    Y_valid_original = Y_train[valid_idx]

    if row_normalize:
        Y_fit_model = normalize(
            Y_fit_original,
            norm="l2",
            axis=1,
            copy=True
        ).astype(np.float32)

        Y_valid_model = normalize(
            Y_valid_original,
            norm="l2",
            axis=1,
            copy=True
        ).astype(np.float32)
    else:
        Y_fit_model = Y_fit_original
        Y_valid_model = Y_valid_original

    target_svd = TruncatedSVD(
        n_components=n_components,
        algorithm="randomized",
        n_iter=7,
        random_state=42
    )

    start_time = time.time()

    Z_fit = target_svd.fit_transform(
        Y_fit_model
    ).astype(np.float32)

    svd_minutes = (
        time.time() - start_time
    ) / 60

    print(f"SVD time: {svd_minutes:.1f} minutes")
    print(
        "Explained variance:",
        target_svd.explained_variance_ratio_.sum()
    )

    # Target-SVD reconstruction ceiling
    Z_valid_oracle = target_svd.transform(
        Y_valid_model
    ).astype(np.float32)

    oracle_raw, oracle_clipped = (
        evaluate_latent_predictions(
            Y_valid_original,
            Z_valid_oracle,
            target_svd,
            batch_size=256
        )
    )

    # Ridge
    ridge = make_pipeline(
        StandardScaler(),
        Ridge(
            alpha=ridge_alpha,
            fit_intercept=True
        )
    )

    ridge.fit(
        X_train_svd[fit_idx],
        Z_fit
    )

    Z_valid_pred = ridge.predict(
        X_train_svd[valid_idx]
    ).astype(np.float32)

    ridge_raw, ridge_clipped = (
        evaluate_latent_predictions(
            Y_valid_original,
            Z_valid_pred,
            target_svd,
            batch_size=256
        )
    )

    result = {
        "components": n_components,
        "row_normalized": row_normalize,
        "explained_variance":
            target_svd.explained_variance_ratio_.sum(),
        "oracle_raw": oracle_raw,
        "oracle_clipped": oracle_clipped,
        "ridge_raw": ridge_raw,
        "ridge_clipped": ridge_clipped,
        "svd_minutes": svd_minutes
    }

    print("\nReconstruction ceiling:")
    print(f"Raw:     {oracle_raw:.6f}")
    print(f"Clipped: {oracle_clipped:.6f}")

    print("\nRidge:")
    print(f"Raw:     {ridge_raw:.6f}")
    print(f"Clipped: {ridge_clipped:.6f}")

    del (
        Z_fit,
        Z_valid_oracle,
        Z_valid_pred,
        Y_fit_model,
        Y_valid_model
    )
    gc.collect()

    return result, target_svd, ridge

## 2.1 128 SVD + normalization

In [ ]:
result_norm128, target_svd_norm128, ridge_norm128 = (
    run_target_experiment(
        n_components=128,
        row_normalize=True,
        ridge_alpha=10.0
    )
)

## 2.2 256 SVD + normalization

In [49]:
def rowwise_pearson(y_true, y_pred):
    y_true = y_true.astype(np.float32, copy=False)
    y_pred = y_pred.astype(np.float32, copy=False)

    y_true_centered = (
        y_true - y_true.mean(axis=1, keepdims=True)
    )
    y_pred_centered = (
        y_pred - y_pred.mean(axis=1, keepdims=True)
    )

    numerator = np.sum(
        y_true_centered * y_pred_centered,
        axis=1
    )

    denominator = np.sqrt(
        np.sum(y_true_centered ** 2, axis=1)
        * np.sum(y_pred_centered ** 2, axis=1)
    )

    return np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator),
        where=denominator > 0
    )


def evaluate_latent_predictions(
    y_true_sparse,
    z_pred,
    target_svd,
    batch_size=256
):
    raw_correlations = []
    clipped_correlations = []

    for start in range(
        0,
        y_true_sparse.shape[0],
        batch_size
    ):
        stop = min(
            start + batch_size,
            y_true_sparse.shape[0]
        )

        y_true_batch = (
            y_true_sparse[start:stop]
            .toarray()
            .astype(np.float32)
        )

        y_pred_batch = (
            target_svd
            .inverse_transform(z_pred[start:stop])
            .astype(np.float32)
        )

        raw_correlations.append(
            rowwise_pearson(
                y_true_batch,
                y_pred_batch
            )
        )

        np.maximum(
            y_pred_batch,
            0,
            out=y_pred_batch
        )

        clipped_correlations.append(
            rowwise_pearson(
                y_true_batch,
                y_pred_batch
            )
        )

        del y_true_batch, y_pred_batch
        gc.collect()

    raw_score = np.concatenate(
        raw_correlations
    ).mean()

    clipped_score = np.concatenate(
        clipped_correlations
    ).mean()

    return raw_score, clipped_score

In [54]:
from pathlib import Path
import numpy as np

file_name = "X_train_multi_svd128.npy"

matches = sorted(
    Path("/kaggle/input").rglob(file_name)
)

print("Found files:")
for path in matches:
    print(path)

if len(matches) == 0:
    raise FileNotFoundError(
        f"在 /kaggle/input 中没有找到 {file_name}。"
        "请确认 Dataset 已添加到当前 Notebook。"
    )

if len(matches) > 1:
    raise RuntimeError(
        f"找到多个同名文件，请检查上方路径并手动选择：{matches}"
    )

X_train_svd = np.load(
    matches[0],
    mmap_mode="r"
)

print("\nLoaded from:", matches[0])
print("Shape:", X_train_svd.shape)
print("dtype:", X_train_svd.dtype)

assert X_train_svd.shape == (105942, 128)
assert X_train_svd.dtype == np.float32

print("X_train_svd loaded successfully.")

Found files:
/kaggle/input/datasets/huangyufeid/multiome-svd128-backup/X_train_multi_svd128.npy

Loaded from: /kaggle/input/datasets/huangyufeid/multiome-svd128-backup/X_train_multi_svd128.npy
Shape: (105942, 128)
dtype: float32
X_train_svd loaded successfully.


In [56]:
import numpy as np

X_train_svd = np.load(
    "/kaggle/input/datasets/huangyufeid/multiome-svd128-backup/X_train_multi_svd128.npy",
    mmap_mode="r"
)

print("X_train_svd shape:", X_train_svd.shape)
print("dtype:", X_train_svd.dtype)

assert X_train_svd.shape == (105942, 128)

X_train_svd shape: (105942, 128)
dtype: float32


In [57]:
required_variables = [
    "Y_train",
    "X_train_svd",
    "fit_idx",
    "valid_idx",
    "evaluate_latent_predictions",
    "run_target_experiment"
]

for name in required_variables:
    print(
        f"{name:30s}",
        "OK" if name in globals() else "MISSING"
    )

Y_train                        OK
X_train_svd                    OK
fit_idx                        OK
valid_idx                      OK
evaluate_latent_predictions    OK
run_target_experiment          OK


In [58]:
result_norm256, target_svd_norm256, ridge_norm256 = (
    run_target_experiment(
        n_components=256,
        row_normalize=True,
        ridge_alpha=10.0
    )
)

Target components: 256, row normalization: True
SVD time: 10.6 minutes
Explained variance: 0.113915935

Reconstruction ceiling:
Raw:     0.637075
Clipped: 0.637827

Ridge:
Raw:     0.594287
Clipped: 0.594381


In [ ]:
comparison = pd.DataFrame([
    {
        "configuration": "Raw target, 128",
        "oracle_clipped": 0.629783,
        "ridge_clipped": 0.594366
    },
    {
        "configuration": "L2 target, 128",
        "oracle_clipped":
            result_norm128["oracle_clipped"],
        "ridge_clipped":
            result_norm128["ridge_clipped"]
    },
    {
        "configuration": "L2 target, 256",
        "oracle_clipped":
            result_norm256["oracle_clipped"],
        "ridge_clipped":
            result_norm256["ridge_clipped"]
    }
])

comparison["model_gap"] = (
    comparison["oracle_clipped"]
    - comparison["ridge_clipped"]
)

comparison["gain_over_mean"] = (
    comparison["ridge_clipped"]
    - mean_baseline_score
)

display(comparison)

## 3.1 Change Ridge into MLP (under 128 SVD)

In [61]:
required = [
    "Y_train",
    "X_train_svd",
    "fit_idx",
    "valid_idx",
    "target_svd_norm256",
    "result_norm256",
    "evaluate_latent_predictions"
]

for name in required:
    print(
        f"{name:30s}",
        "OK" if name in globals() else "MISSING"
    )

Y_train                        OK
X_train_svd                    OK
fit_idx                        OK
valid_idx                      OK
target_svd_norm256             OK
result_norm256                 OK
evaluate_latent_predictions    OK


In [63]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 清理之前可能残留的Keras状态，并固定随机种子
keras.backend.clear_session()
keras.utils.set_random_seed(42)

# 输入：128维ATAC latent
# 输出：256维标准化后的Target-SVD latent
mlp_model = keras.Sequential([
    keras.Input(shape=(X_fit_mlp.shape[1],)),

    layers.Dense(
        512,
        activation="relu"
    ),
    layers.Dropout(0.10),

    layers.Dense(
        256,
        activation="relu"
    ),
    layers.Dropout(0.10),

    layers.Dense(
        Z_fit_mlp.shape[1],
        activation="linear"
    )
])

mlp_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="mse"
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)

mlp_model.summary()

history = mlp_model.fit(
    X_fit_mlp,
    Z_fit_mlp,
    validation_data=(
        X_valid_mlp,
        Z_valid_mlp
    ),
    epochs=50,
    batch_size=1024,
    callbacks=[
        early_stopping,
        reduce_lr
    ],
    shuffle=True,
    verbose=1
)

2026-10-03 05:35:57.002692: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │        65,792 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 263,168 (1.00 MB)

 Trainable params: 263,168 (1.00 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - loss: 1.0075 - val_loss: 1.0315 - learning_rate: 0.0010
Epoch 2/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.9980 - val_loss: 1.0183 - learning_rate: 0.0010
Epoch 3/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.9863 - val_loss: 0.9621 - learning_rate: 0.0010
Epoch 4/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.9702 - val_loss: 0.9338 - learning_rate: 0.0010
Epoch 5/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - loss: 0.9579 - val_loss: 0.9181 - learning_rate: 0.0010
Epoch 6/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - loss: 0.9505 - val_loss: 0.9092 - learning_rate: 0.0010
Epoch 7/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.9455 - val_loss: 0.9021 - learning_rate: 0.0010
Epoch 8/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.9416 - val_loss: 0.8967 - learning_rate: 0.0010
Epoch 9/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - loss: 0.9375 - val_loss: 0.8934 - learning_rate: 0.0010
Epoch 10/50
56/56 ━

In [66]:
from sklearn.preprocessing import StandardScaler, normalize
import numpy as np
import gc

# 1. 标准化128维ATAC latent
x_scaler_mlp = StandardScaler()

X_fit_mlp = x_scaler_mlp.fit_transform(
    X_train_svd[fit_idx]
).astype(np.float32)

X_valid_mlp = x_scaler_mlp.transform(
    X_train_svd[valid_idx]
).astype(np.float32)


# 2. 对GEX进行与之前相同的L2 row normalization
Y_fit_normalized = normalize(
    Y_train[fit_idx],
    norm="l2",
    axis=1,
    copy=True
).astype(np.float32)

Y_valid_normalized = normalize(
    Y_train[valid_idx],
    norm="l2",
    axis=1,
    copy=True
).astype(np.float32)


# 3. 使用已经训练好的256维Target SVD
Z_fit_256 = target_svd_norm256.transform(
    Y_fit_normalized
).astype(np.float32)

Z_valid_256 = target_svd_norm256.transform(
    Y_valid_normalized
).astype(np.float32)


# 4. 标准化每一个target latent component
# 否则方差最大的前几个成分会主导MSE
z_scaler_mlp = StandardScaler()

Z_fit_mlp = z_scaler_mlp.fit_transform(
    Z_fit_256
).astype(np.float32)

Z_valid_mlp = z_scaler_mlp.transform(
    Z_valid_256
).astype(np.float32)


print("X_fit_mlp:  ", X_fit_mlp.shape)
print("X_valid_mlp:", X_valid_mlp.shape)
print("Z_fit_mlp:  ", Z_fit_mlp.shape)
print("Z_valid_mlp:", Z_valid_mlp.shape)

assert X_fit_mlp.shape == (len(fit_idx), 128)
assert X_valid_mlp.shape == (len(valid_idx), 128)
assert Z_fit_mlp.shape == (len(fit_idx), 256)
assert Z_valid_mlp.shape == (len(valid_idx), 256)

del Y_fit_normalized, Y_valid_normalized
gc.collect()

X_fit_mlp:   (57119, 128)
X_valid_mlp: (6960, 128)
Z_fit_mlp:   (57119, 256)
Z_valid_mlp: (6960, 256)


154

In [67]:
Z_valid_pred_scaled = mlp_model.predict(
    X_valid_mlp,
    batch_size=1024,
    verbose=1
).astype(np.float32)

# 从标准化latent恢复成Target-SVD的原始latent尺度
Z_valid_pred = z_scaler_mlp.inverse_transform(
    Z_valid_pred_scaled
).astype(np.float32)

print("Prediction shape:", Z_valid_pred.shape)

assert Z_valid_pred.shape == (
    len(valid_idx),
    256
)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
Prediction shape: (6960, 256)


In [69]:
import numpy as np


def evaluate_fixed_prediction(
    y_true_sparse,
    prediction_vector,
    batch_size=256
):
    correlations = []

    for start in range(
        0,
        y_true_sparse.shape[0],
        batch_size
    ):
        stop = min(
            start + batch_size,
            y_true_sparse.shape[0]
        )

        y_true_batch = (
            y_true_sparse[start:stop]
            .toarray()
            .astype(np.float32)
        )

        y_pred_batch = np.broadcast_to(
            prediction_vector,
            y_true_batch.shape
        )

        correlations.append(
            rowwise_pearson(
                y_true_batch,
                y_pred_batch
            )
        )

    return np.concatenate(
        correlations
    ).mean()


# 只用训练子集计算每个基因的平均表达
gene_mean = np.asarray(
    Y_train[fit_idx].mean(axis=0)
).ravel().astype(np.float32)

# 在严格验证集上评价
mean_baseline_score = evaluate_fixed_prediction(
    Y_train[valid_idx],
    gene_mean,
    batch_size=256
)

print(
    "Gene-mean baseline:",
    f"{mean_baseline_score:.6f}"
)

Gene-mean baseline: 0.560037


In [70]:
mlp_raw, mlp_clipped = evaluate_latent_predictions(
    Y_train[valid_idx],
    Z_valid_pred,
    target_svd_norm256,
    batch_size=256
)

ridge_256_score = result_norm256[
    "ridge_clipped"
]

print("Strict donor + time validation")
print("--------------------------------")
print(
    f"Gene-mean baseline: {mean_baseline_score:.6f}"
)
print(
    f"Ridge-256:          {ridge_256_score:.6f}"
)
print(
    f"MLP raw:            {mlp_raw:.6f}"
)
print(
    f"MLP clipped:        {mlp_clipped:.6f}"
)

print(
    "\nMLP improvement over Ridge:",
    f"{mlp_clipped - ridge_256_score:.6f}"
)

Strict donor + time validation
--------------------------------
Gene-mean baseline: 0.560037
Ridge-256:          0.594381
MLP raw:            0.597296
MLP clipped:        0.597377

MLP improvement over Ridge: 0.002996


In [71]:
history_df = pd.DataFrame(history.history)

best_epoch = (
    history_df["val_loss"].idxmin() + 1
)

best_train_loss = history_df.loc[
    best_epoch - 1,
    "loss"
]

best_valid_loss = history_df.loc[
    best_epoch - 1,
    "val_loss"
]

print("Best epoch:", best_epoch)
print(
    "Train loss at best epoch:",
    f"{best_train_loss:.6f}"
)
print(
    "Valid loss at best epoch:",
    f"{best_valid_loss:.6f}"
)

Best epoch: 23
Train loss at best epoch: 0.911692
Valid loss at best epoch: 0.875673


In [72]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import pandas as pd
import numpy as np
import gc

# 使用和MLP相同的训练/验证输入
ridge_x_scaler = StandardScaler()

X_fit_ridge = ridge_x_scaler.fit_transform(
    X_train_svd[fit_idx]
).astype(np.float32)

X_valid_ridge = ridge_x_scaler.transform(
    X_train_svd[valid_idx]
).astype(np.float32)

alphas = [
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    1000.0
]

ridge_results = []

for alpha in alphas:
    print(f"\nTesting alpha={alpha}")

    model = Ridge(
        alpha=alpha,
        fit_intercept=True
    )

    model.fit(
        X_fit_ridge,
        Z_fit_256
    )

    Z_valid_pred_ridge = model.predict(
        X_valid_ridge
    ).astype(np.float32)

    raw_score, clipped_score = (
        evaluate_latent_predictions(
            Y_train[valid_idx],
            Z_valid_pred_ridge,
            target_svd_norm256,
            batch_size=256
        )
    )

    ridge_results.append({
        "alpha": alpha,
        "raw_score": raw_score,
        "clipped_score": clipped_score
    })

    print(f"Raw:     {raw_score:.6f}")
    print(f"Clipped: {clipped_score:.6f}")

    del model, Z_valid_pred_ridge
    gc.collect()

ridge_alpha_results = pd.DataFrame(
    ridge_results
).sort_values(
    "clipped_score",
    ascending=False
).reset_index(drop=True)

display(ridge_alpha_results)

best_alpha = ridge_alpha_results.loc[
    0,
    "alpha"
]

best_ridge_score = ridge_alpha_results.loc[
    0,
    "clipped_score"
]

print("Best alpha:", best_alpha)
print("Best Ridge score:", best_ridge_score)
print(
    "Difference from MLP:",
    mlp_clipped - best_ridge_score
)


Testing alpha=0.01
Raw:     0.594295
Clipped: 0.594389

Testing alpha=0.1
Raw:     0.594295
Clipped: 0.594389

Testing alpha=1.0
Raw:     0.594294
Clipped: 0.594389

Testing alpha=10.0
Raw:     0.594287
Clipped: 0.594381

Testing alpha=100.0
Raw:     0.594214
Clipped: 0.594306

Testing alpha=1000.0
Raw:     0.593484
Clipped: 0.593553


,alpha,raw_score,clipped_score
0,0.01,0.594295,0.594389
1,0.10,0.594295,0.594389
2,1.00,0.594294,0.594389
3,10.00,0.594287,0.594381
4,100.00,0.594214,0.594306
5,1000.00,0.593484,0.593553


Best alpha: 0.01
Best Ridge score: 0.59438944
Difference from MLP: 0.0029873252


In [73]:
import joblib
import numpy as np

joblib.dump(
    target_svd_norm256,
    WORK_DIR / "validation_target_svd256.joblib"
)

ridge_alpha_results.to_csv(
    WORK_DIR / "ridge_alpha_results_x128_y256.csv",
    index=False
)

np.save(
    WORK_DIR / "strict_fit_idx.npy",
    fit_idx,
    allow_pickle=False
)

np.save(
    WORK_DIR / "strict_valid_idx.npy",
    valid_idx,
    allow_pickle=False
)

mlp_model.save(
    WORK_DIR / "validation_mlp_x128_y256.keras"
)

joblib.dump(
    x_scaler_mlp,
    WORK_DIR / "validation_mlp_x_scaler.joblib"
)

joblib.dump(
    z_scaler_mlp,
    WORK_DIR / "validation_mlp_z_scaler.joblib"
)

print("Validation artifacts saved.")

Validation artifacts saved.


In [74]:
import gc

variables_to_remove = [
    "X_fit_mlp",
    "X_valid_mlp",
    "Z_fit_mlp",
    "Z_valid_mlp",
    "Z_fit_256",
    "Z_valid_256",
    "Z_valid_pred",
    "Z_valid_pred_scaled",
    "X_fit_ridge",
    "X_valid_ridge",
    "X_train_svd",
    "X_test_svd"
]

for variable_name in variables_to_remove:
    globals().pop(variable_name, None)

gc.collect()

print("Temporary variables cleared.")

Temporary variables cleared.


In [75]:
from sklearn.decomposition import TruncatedSVD
import scipy.sparse as sp
import numpy as np
import joblib
import time
import gc

X_TRAIN_256_PATH = (
    WORK_DIR / "X_train_multi_svd256.npy"
)

X_TEST_256_PATH = (
    WORK_DIR / "X_test_multi_svd256.npy"
)

INPUT_SVD_256_PATH = (
    WORK_DIR / "input_multi_svd256.joblib"
)

if (
    X_TRAIN_256_PATH.exists()
    and X_TEST_256_PATH.exists()
    and INPUT_SVD_256_PATH.exists()
):
    print("Existing Input SVD256 artifacts found.")

    X_train_svd256 = np.load(
        X_TRAIN_256_PATH,
        mmap_mode="r"
    )

    X_test_svd256 = np.load(
        X_TEST_256_PATH,
        mmap_mode="r"
    )

    input_svd256 = joblib.load(
        INPUT_SVD_256_PATH
    )

else:
    print("Loading sparse ATAC matrices...")

    X_train_sparse = sp.load_npz(
        WORK_DIR / "train_multi_inputs_matrix.npz"
    ).astype(np.float32)

    X_test_sparse = sp.load_npz(
        WORK_DIR / "test_multi_inputs_matrix.npz"
    ).astype(np.float32)

    n_train_cells = X_train_sparse.shape[0]

    print("Train:", X_train_sparse.shape)
    print("Test: ", X_test_sparse.shape)

    print("Combining sparse matrices...")

    X_all_sparse = sp.vstack(
        [X_train_sparse, X_test_sparse],
        format="csr"
    )

    input_svd256 = TruncatedSVD(
        n_components=256,
        algorithm="randomized",
        n_iter=5,
        random_state=42
    )

    start_time = time.time()

    X_all_svd256 = input_svd256.fit_transform(
        X_all_sparse
    ).astype(np.float32)

    elapsed = (
        time.time() - start_time
    ) / 60

    print("SVD output:", X_all_svd256.shape)
    print(f"Elapsed: {elapsed:.1f} minutes")
    print(
        "Explained variance:",
        input_svd256
        .explained_variance_ratio_
        .sum()
    )

    X_train_svd256 = X_all_svd256[
        :n_train_cells
    ]

    X_test_svd256 = X_all_svd256[
        n_train_cells:
    ]

    assert X_train_svd256.shape == (
        105942,
        256
    )

    assert X_test_svd256.shape == (
        55935,
        256
    )

    np.save(
        X_TRAIN_256_PATH,
        X_train_svd256,
        allow_pickle=False
    )

    np.save(
        X_TEST_256_PATH,
        X_test_svd256,
        allow_pickle=False
    )

    joblib.dump(
        input_svd256,
        INPUT_SVD_256_PATH
    )

    print("Input SVD256 artifacts saved.")

    del (
        X_all_svd256,
        X_all_sparse,
        X_train_sparse,
        X_test_sparse
    )

    gc.collect()

print("X_train_svd256:", X_train_svd256.shape)
print("X_test_svd256: ", X_test_svd256.shape)

Existing Input SVD256 artifacts found.
X_train_svd256: (105942, 256)
X_test_svd256:  (55935, 256)


In [77]:
if "Y_train" not in globals():
    Y_train = sp.load_npz(
        WORK_DIR / "train_multi_targets_matrix.npz"
    ).astype(np.float32)

if "target_svd_norm256" not in globals():
    target_svd_norm256 = joblib.load(
        WORK_DIR / "validation_target_svd256.joblib"
    )

In [78]:
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.linear_model import Ridge

Y_fit_normalized = normalize(
    Y_train[fit_idx],
    norm="l2",
    axis=1,
    copy=True
).astype(np.float32)

Z_fit_target256 = (
    target_svd_norm256
    .transform(Y_fit_normalized)
    .astype(np.float32)
)

del Y_fit_normalized
gc.collect()

22

In [79]:
x_scaler_256 = StandardScaler()

X_fit_256 = x_scaler_256.fit_transform(
    X_train_svd256[fit_idx]
).astype(np.float32)

X_valid_256 = x_scaler_256.transform(
    X_train_svd256[valid_idx]
).astype(np.float32)

ridge_x256 = Ridge(
    alpha=1.0,
    fit_intercept=True
)

ridge_x256.fit(
    X_fit_256,
    Z_fit_target256
)

Z_valid_pred_x256 = ridge_x256.predict(
    X_valid_256
).astype(np.float32)

ridge_x256_raw, ridge_x256_clipped = (
    evaluate_latent_predictions(
        Y_train[valid_idx],
        Z_valid_pred_x256,
        target_svd_norm256,
        batch_size=256
    )
)

print("Strict donor + time validation")
print("--------------------------------")
print(f"Gene mean:       {mean_baseline_score:.6f}")
print(f"Ridge X128:      {best_ridge_score:.6f}")
print(f"MLP X128:        {mlp_clipped:.6f}")
print(f"Ridge X256 raw:  {ridge_x256_raw:.6f}")
print(f"Ridge X256 clip: {ridge_x256_clipped:.6f}")

print(
    "\nX256 improvement over X128:",
    f"{ridge_x256_clipped - best_ridge_score:.6f}"
)

Strict donor + time validation
--------------------------------
Gene mean:       0.560037
Ridge X128:      0.594389
MLP X128:        0.597377
Ridge X256 raw:  0.594383
Ridge X256 clip: 0.594472

X256 improvement over X128: 0.000082


## 3 domain shift

In [81]:
import numpy as np
import scipy.sparse as sp
import pandas as pd

X_train_svd128 = np.load(
    "/kaggle/input/datasets/huangyufeid/multiome-svd128-backup/X_train_multi_svd128.npy",
    mmap_mode="r"
)

if "Y_train" not in globals():
    Y_train = sp.load_npz(
        WORK_DIR / "train_multi_targets_matrix.npz"
    ).astype(np.float32)

if "train_meta" not in globals():
    train_meta = pd.read_csv(
        WORK_DIR / "train_multi_metadata.csv",
        dtype={"donor": str}
    )

print("X:", X_train_svd128.shape)
print("Y:", Y_train.shape)
print("Metadata:", train_meta.shape)

X: (105942, 128)
Y: (105942, 23418)
Metadata: (105942, 7)


In [82]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

import time
import gc


def evaluate_domain_split(
    split_name,
    fit_mask,
    valid_mask,
    n_target_components=128,
    alpha=1.0
):
    fit_indices = np.flatnonzero(fit_mask)
    valid_indices = np.flatnonzero(valid_mask)

    print("\n" + "=" * 60)
    print(split_name)
    print("Train cells:", len(fit_indices))
    print("Valid cells:", len(valid_indices))

    # 只在当前split的训练target上拟合SVD
    target_svd = TruncatedSVD(
        n_components=n_target_components,
        algorithm="randomized",
        n_iter=7,
        random_state=42
    )

    start_time = time.time()

    Z_fit = target_svd.fit_transform(
        Y_train[fit_indices]
    ).astype(np.float32)

    print(
        "Target SVD time:",
        f"{(time.time() - start_time) / 60:.1f} minutes"
    )

    Y_valid_split = Y_train[valid_indices]

    # Target-SVD重构分数
    Z_valid_oracle = target_svd.transform(
        Y_valid_split
    ).astype(np.float32)

    oracle_raw, oracle_clipped = (
        evaluate_latent_predictions(
            Y_valid_split,
            Z_valid_oracle,
            target_svd,
            batch_size=256
        )
    )

    # 输入标准化
    scaler = StandardScaler()

    X_fit = scaler.fit_transform(
        X_train_svd128[fit_indices]
    ).astype(np.float32)

    X_valid = scaler.transform(
        X_train_svd128[valid_indices]
    ).astype(np.float32)

    # Ridge
    ridge = Ridge(
        alpha=alpha,
        fit_intercept=True
    )

    ridge.fit(X_fit, Z_fit)

    Z_valid_pred = ridge.predict(
        X_valid
    ).astype(np.float32)

    ridge_raw, ridge_clipped = (
        evaluate_latent_predictions(
            Y_valid_split,
            Z_valid_pred,
            target_svd,
            batch_size=256
        )
    )

    result = {
        "split": split_name,
        "n_train": len(fit_indices),
        "n_valid": len(valid_indices),
        "oracle_clipped": oracle_clipped,
        "ridge_raw": ridge_raw,
        "ridge_clipped": ridge_clipped,
        "model_gap":
            oracle_clipped - ridge_clipped
    }

    print("Oracle:", f"{oracle_clipped:.6f}")
    print("Ridge: ", f"{ridge_clipped:.6f}")
    print(
        "Gap:   ",
        f"{oracle_clipped - ridge_clipped:.6f}"
    )

    del (
        Z_fit,
        Z_valid_oracle,
        Z_valid_pred,
        X_fit,
        X_valid,
        Y_valid_split,
        target_svd,
        scaler,
        ridge
    )

    gc.collect()

    return result

In [83]:
donor = train_meta["donor"].astype(str)
day = train_meta["day"].astype(int)

VALID_DONOR = "32606"
VALID_DAY = 7

# 1. 新donor，但允许模型见过day 7
donor_fit_mask = (
    donor != VALID_DONOR
).to_numpy()

donor_valid_mask = (
    donor == VALID_DONOR
).to_numpy()


# 2. 新时间，但允许模型见过donor 32606
time_fit_mask = (
    day != VALID_DAY
).to_numpy()

time_valid_mask = (
    day == VALID_DAY
).to_numpy()


# 3. 同时新donor和新时间
joint_fit_mask = (
    (donor != VALID_DONOR)
    & (day != VALID_DAY)
).to_numpy()

joint_valid_mask = (
    (donor == VALID_DONOR)
    & (day == VALID_DAY)
).to_numpy()

In [84]:
domain_results = []

domain_results.append(
    evaluate_domain_split(
        "Donor-only",
        donor_fit_mask,
        donor_valid_mask
    )
)

domain_results.append(
    evaluate_domain_split(
        "Time-only",
        time_fit_mask,
        time_valid_mask
    )
)

domain_results.append(
    evaluate_domain_split(
        "Donor + Time",
        joint_fit_mask,
        joint_valid_mask
    )
)

domain_results_df = pd.DataFrame(
    domain_results
)

display(domain_results_df)

domain_results_df.to_csv(
    WORK_DIR / "multiome_domain_validation.csv",
    index=False
)


Donor-only
Train cells: 72241
Valid cells: 33701
Target SVD time: 6.6 minutes
Oracle: 0.681046
Ridge:  0.663116
Gap:    0.017930

Time-only
Train cells: 83860
Valid cells: 22082
Target SVD time: 7.9 minutes
Oracle: 0.636845
Ridge:  0.601190
Gap:    0.035655

Donor + Time
Train cells: 57119
Valid cells: 6960
Target SVD time: 5.2 minutes
Oracle: 0.629783
Ridge:  0.594374
Gap:    0.035409


,split,n_train,n_valid,oracle_clipped,ridge_raw,ridge_clipped,model_gap
0,Donor-only,72241,33701,0.681046,0.663046,0.663116,0.017930
1,Time-only,83860,22082,0.636845,0.601053,0.601190,0.035655
2,Donor + Time,57119,6960,0.629783,0.594255,0.594374,0.035409


Prediting time is hard

In [85]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

import numpy as np
import gc
import time

day_array = (
    train_meta["day"]
    .astype(float)
    .to_numpy()
)

time_fit_indices = np.flatnonzero(
    time_fit_mask
)

time_valid_indices = np.flatnonzero(
    time_valid_mask
)

print("Training cells:", len(time_fit_indices))
print("Validation cells:", len(time_valid_indices))

Training cells: 83860
Validation cells: 22082


In [86]:
target_svd_time_day = TruncatedSVD(
    n_components=128,
    algorithm="randomized",
    n_iter=7,
    random_state=42
)

Z_fit_time_day = (
    target_svd_time_day
    .fit_transform(
        Y_train[time_fit_indices]
    )
    .astype(np.float32)
)

In [87]:
X_fit_time_day = np.column_stack([
    np.asarray(
        X_train_svd128[time_fit_indices],
        dtype=np.float32
    ),
    day_array[time_fit_indices]
]).astype(np.float32)

X_valid_time_day = np.column_stack([
    np.asarray(
        X_train_svd128[time_valid_indices],
        dtype=np.float32
    ),
    day_array[time_valid_indices]
]).astype(np.float32)

print("Augmented input:", X_fit_time_day.shape)

Augmented input: (83860, 129)


In [88]:
time_day_scaler = StandardScaler()

X_fit_time_day = (
    time_day_scaler
    .fit_transform(X_fit_time_day)
    .astype(np.float32)
)

X_valid_time_day = (
    time_day_scaler
    .transform(X_valid_time_day)
    .astype(np.float32)
)

ridge_time_day = Ridge(
    alpha=1.0,
    fit_intercept=True
)

ridge_time_day.fit(
    X_fit_time_day,
    Z_fit_time_day
)

Z_valid_pred_time_day = (
    ridge_time_day
    .predict(X_valid_time_day)
    .astype(np.float32)
)

In [89]:
time_day_raw, time_day_clipped = (
    evaluate_latent_predictions(
        Y_train[time_valid_indices],
        Z_valid_pred_time_day,
        target_svd_time_day,
        batch_size=256
    )
)

print("Time-only comparison")
print("--------------------")
print(f"Without day: {0.601190:.6f}")
print(f"With day:    {time_day_clipped:.6f}")
print(
    "Improvement:",
    f"{time_day_clipped - 0.601190:.6f}"
)

Time-only comparison
--------------------
Without day: 0.601190
With day:    0.590148
Improvement: -0.011042


## 4.1 time-only MLP

In [90]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# 不加入day
x_scaler_time_mlp = StandardScaler()

X_fit_time_mlp = (
    x_scaler_time_mlp
    .fit_transform(
        X_train_svd128[time_fit_indices]
    )
    .astype(np.float32)
)

X_valid_time_mlp = (
    x_scaler_time_mlp
    .transform(
        X_train_svd128[time_valid_indices]
    )
    .astype(np.float32)
)

# Z_fit_time_day本身不包含day；
# 它只是前面变量命名中带有time_day
Z_fit_time = Z_fit_time_day

Z_valid_time = (
    target_svd_time_day
    .transform(
        Y_train[time_valid_indices]
    )
    .astype(np.float32)
)

z_scaler_time_mlp = StandardScaler()

Z_fit_time_mlp = (
    z_scaler_time_mlp
    .fit_transform(Z_fit_time)
    .astype(np.float32)
)

Z_valid_time_mlp = (
    z_scaler_time_mlp
    .transform(Z_valid_time)
    .astype(np.float32)
)

print("X train:", X_fit_time_mlp.shape)
print("X valid:", X_valid_time_mlp.shape)
print("Z train:", Z_fit_time_mlp.shape)
print("Z valid:", Z_valid_time_mlp.shape)

X train: (83860, 128)
X valid: (22082, 128)
Z train: (83860, 128)
Z valid: (22082, 128)


In [91]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

keras.backend.clear_session()
keras.utils.set_random_seed(42)

inputs = keras.Input(
    shape=(128,)
)

x = layers.Dense(
    256,
    activation="swish",
    kernel_regularizer=regularizers.l2(1e-5)
)(inputs)

x = layers.Dropout(0.10)(x)

x = layers.Dense(
    256,
    activation="swish",
    kernel_regularizer=regularizers.l2(1e-5)
)(x)

x = layers.Dropout(0.10)(x)

outputs = layers.Dense(
    128,
    activation="linear"
)(x)

mlp_time = keras.Model(
    inputs,
    outputs
)

mlp_time.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="mse"
)

In [92]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )
]

history_time = mlp_time.fit(
    X_fit_time_mlp,
    Z_fit_time_mlp,

    validation_data=(
        X_valid_time_mlp,
        Z_valid_time_mlp
    ),

    epochs=50,
    batch_size=1024,
    shuffle=True,

    callbacks=callbacks,
    verbose=2
)

Epoch 1/50
82/82 - 4s - 44ms/step - loss: 0.9569 - val_loss: 1.0069 - learning_rate: 0.0010
Epoch 2/50
82/82 - 2s - 22ms/step - loss: 0.8904 - val_loss: 0.9527 - learning_rate: 0.0010
Epoch 3/50
82/82 - 2s - 23ms/step - loss: 0.8752 - val_loss: 0.9343 - learning_rate: 0.0010
Epoch 4/50
82/82 - 2s - 21ms/step - loss: 0.8652 - val_loss: 0.9224 - learning_rate: 0.0010
Epoch 5/50
82/82 - 2s - 21ms/step - loss: 0.8579 - val_loss: 0.9131 - learning_rate: 0.0010
Epoch 6/50
82/82 - 2s - 21ms/step - loss: 0.8522 - val_loss: 0.9088 - learning_rate: 0.0010
Epoch 7/50
82/82 - 2s - 21ms/step - loss: 0.8479 - val_loss: 0.9022 - learning_rate: 0.0010
Epoch 8/50
82/82 - 2s - 21ms/step - loss: 0.8445 - val_loss: 0.9030 - learning_rate: 0.0010
Epoch 9/50
82/82 - 2s - 22ms/step - loss: 0.8420 - val_loss: 0.9022 - learning_rate: 0.0010
Epoch 10/50
82/82 - 2s - 21ms/step - loss: 0.8398 - val_loss: 0.9014 - learning_rate: 0.0010
Epoch 11/50
82/82 - 2s - 21ms/step - loss: 0.8378 - val_loss: 0.9010 - learning

In [93]:
Z_valid_pred_time_scaled = (
    mlp_time.predict(
        X_valid_time_mlp,
        batch_size=1024,
        verbose=1
    )
    .astype(np.float32)
)

Z_valid_pred_time_mlp = (
    z_scaler_time_mlp
    .inverse_transform(
        Z_valid_pred_time_scaled
    )
    .astype(np.float32)
)

time_mlp_raw, time_mlp_clipped = (
    evaluate_latent_predictions(
        Y_train[time_valid_indices],
        Z_valid_pred_time_mlp,
        target_svd_time_day,
        batch_size=256
    )
)

print("Time-only model comparison")
print("--------------------------")
print(f"Ridge: {0.601190:.6f}")
print(f"MLP:   {time_mlp_clipped:.6f}")
print(
    "Improvement:",
    f"{time_mlp_clipped - 0.601190:.6f}"
)

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
Time-only model comparison
--------------------------
Ridge: 0.601190
MLP:   0.602988
Improvement: 0.001798


## combine ridge & MLP

In [96]:
from pathlib import Path
import numpy as np

matches = sorted(
    Path("/kaggle/input").rglob(
        "X_train_multi_svd128.npy"
    )
)

if len(matches) == 0:
    raise FileNotFoundError(
        "没有在 /kaggle/input 中找到 "
        "X_train_multi_svd128.npy"
    )

print("Loading:", matches[0])

X_train_svd = np.load(
    matches[0],
    mmap_mode="r"
)

print("X_train_svd:", X_train_svd.shape)

assert X_train_svd.shape == (105942, 128)

Loading: /kaggle/input/datasets/huangyufeid/multiome-svd128-backup/X_train_multi_svd128.npy
X_train_svd: (105942, 128)


In [98]:
import numpy as np

required = [
    "mlp_model",
    "x_scaler_mlp",
    "z_scaler_mlp",
    "ridge_norm256",
    "X_train_svd"
]

for name in required:
    print(
        f"{name:20s}",
        "OK" if name in globals() else "MISSING"
    )

mlp_model            OK
x_scaler_mlp         OK
z_scaler_mlp         OK
ridge_norm256        OK
X_train_svd          OK


In [100]:
import numpy as np
import pandas as pd
import gc


# ============================================================
# 1. 生成Ridge的验证集latent预测
# ============================================================

Z_valid_pred_ridge = ridge_norm256.predict(
    X_train_svd[valid_idx]
).astype(np.float32)


# ============================================================
# 2. 生成MLP的验证集latent预测
# ============================================================

X_valid_mlp_blend = x_scaler_mlp.transform(
    X_train_svd[valid_idx]
).astype(np.float32)

Z_valid_pred_mlp_scaled = mlp_model.predict(
    X_valid_mlp_blend,
    batch_size=1024,
    verbose=1
).astype(np.float32)

# 从标准化latent恢复到原始Target-SVD latent尺度
Z_valid_pred_mlp = z_scaler_mlp.inverse_transform(
    Z_valid_pred_mlp_scaled
).astype(np.float32)


print(
    "Ridge latent prediction:",
    Z_valid_pred_ridge.shape
)
print(
    "MLP latent prediction:  ",
    Z_valid_pred_mlp.shape
)

assert Z_valid_pred_ridge.shape == (
    len(valid_idx),
    256
)

assert Z_valid_pred_mlp.shape == (
    len(valid_idx),
    256
)


# ============================================================
# 3. 测试不同的Ridge–MLP融合权重
# ============================================================

blend_records = []

for mlp_weight in np.linspace(0.0, 1.0, 11):
    ridge_weight = 1.0 - mlp_weight

    Z_valid_pred_blend = (
        ridge_weight * Z_valid_pred_ridge
        + mlp_weight * Z_valid_pred_mlp
    ).astype(np.float32)

    raw_score, clipped_score = (
        evaluate_latent_predictions(
            Y_train[valid_idx],
            Z_valid_pred_blend,
            target_svd_norm256,
            batch_size=256
        )
    )

    blend_records.append({
        "ridge_weight": ridge_weight,
        "mlp_weight": mlp_weight,
        "raw_score": raw_score,
        "clipped_score": clipped_score
    })

    print(
        f"Ridge={ridge_weight:.1f}, "
        f"MLP={mlp_weight:.1f} | "
        f"Raw={raw_score:.6f}, "
        f"Clipped={clipped_score:.6f}"
    )

    del Z_valid_pred_blend
    gc.collect()


# ============================================================
# 4. 整理结果
# ============================================================

blend_results = pd.DataFrame(blend_records)

display(blend_results)

best_index = blend_results[
    "clipped_score"
].idxmax()

best_row = blend_results.loc[best_index]

pure_ridge_score = blend_results.loc[
    np.isclose(
        blend_results["mlp_weight"],
        0.0
    ),
    "clipped_score"
].iloc[0]

pure_mlp_score = blend_results.loc[
    np.isclose(
        blend_results["mlp_weight"],
        1.0
    ),
    "clipped_score"
].iloc[0]


print("\nBest blend")
print("--------------------------------")
print(
    f"Ridge weight: {best_row['ridge_weight']:.1f}"
)
print(
    f"MLP weight:   {best_row['mlp_weight']:.1f}"
)
print(
    f"Best score:   {best_row['clipped_score']:.6f}"
)

print("\nSingle-model scores")
print("--------------------------------")
print(f"Pure Ridge:   {pure_ridge_score:.6f}")
print(f"Pure MLP:     {pure_mlp_score:.6f}")

print(
    "Gain over Ridge:",
    f"{best_row['clipped_score'] - pure_ridge_score:.6f}"
)

print(
    "Gain over MLP:",
    f"{best_row['clipped_score'] - pure_mlp_score:.6f}"
)

print(
    "Gain over best single model:",
    f"{best_row['clipped_score'] - max(pure_ridge_score, pure_mlp_score):.6f}"
)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
Ridge latent prediction: (6960, 256)
MLP latent prediction:   (6960, 256)
Ridge=1.0, MLP=0.0 | Raw=0.594287, Clipped=0.594381
Ridge=0.9, MLP=0.1 | Raw=0.594919, Clipped=0.595002
Ridge=0.8, MLP=0.2 | Raw=0.595480, Clipped=0.595553
Ridge=0.7, MLP=0.3 | Raw=0.595967, Clipped=0.596034
Ridge=0.6, MLP=0.4 | Raw=0.596381, Clipped=0.596443
Ridge=0.5, MLP=0.5 | Raw=0.596720, Clipped=0.596780
Ridge=0.4, MLP=0.6 | Raw=0.596985, Clipped=0.597044
Ridge=0.3, MLP=0.7 | Raw=0.597176, Clipped=0.597237
Ridge=0.2, MLP=0.8 | Raw=0.597291, Clipped=0.597356
Ridge=0.1, MLP=0.9 | Raw=0.597331, Clipped=0.597403
Ridge=0.0, MLP=1.0 | Raw=0.597296, Clipped=0.597377


,ridge_weight,mlp_weight,raw_score,clipped_score
0,1.0,0.0,0.594287,0.594381
1,0.9,0.1,0.594919,0.595002
2,0.8,0.2,0.595480,0.595553
3,0.7,0.3,0.595967,0.596034
4,0.6,0.4,0.596381,0.596443
5,0.5,0.5,0.596720,0.596780
6,0.4,0.6,0.596985,0.597044
7,0.3,0.7,0.597176,0.597237
8,0.2,0.8,0.597291,0.597356
9,0.1,0.9,0.597331,0.597403



Best blend
--------------------------------
Ridge weight: 0.1
MLP weight:   0.9
Best score:   0.597403

Single-model scores
--------------------------------
Pure Ridge:   0.594381
Pure MLP:     0.597377
Gain over Ridge: 0.003021
Gain over MLP: 0.000026
Gain over best single model: 0.000026


Different seeds on MLP

In [103]:
import numpy as np
import gc

from sklearn.preprocessing import normalize


# ============================================================
# 1. 恢复MLP输入
# 使用已经拟合好的x_scaler_mlp
# ============================================================

X_fit_mlp = x_scaler_mlp.transform(
    X_train_svd[fit_idx]
).astype(np.float32)

X_valid_mlp = x_scaler_mlp.transform(
    X_train_svd[valid_idx]
).astype(np.float32)


# ============================================================
# 2. 恢复归一化后的GEX目标
# ============================================================

Y_fit_normalized = normalize(
    Y_train[fit_idx],
    norm="l2",
    axis=1,
    copy=True
).astype(np.float32)

Y_valid_normalized = normalize(
    Y_train[valid_idx],
    norm="l2",
    axis=1,
    copy=True
).astype(np.float32)


# ============================================================
# 3. 转换到256维Target-SVD空间
# ============================================================

Z_fit_256 = target_svd_norm256.transform(
    Y_fit_normalized
).astype(np.float32)

Z_valid_256 = target_svd_norm256.transform(
    Y_valid_normalized
).astype(np.float32)


# ============================================================
# 4. 使用已经拟合好的z_scaler_mlp标准化latent目标
# ============================================================

Z_fit_mlp = z_scaler_mlp.transform(
    Z_fit_256
).astype(np.float32)

Z_valid_mlp = z_scaler_mlp.transform(
    Z_valid_256
).astype(np.float32)


# ============================================================
# 5. 检查
# ============================================================

print("X_fit_mlp:  ", X_fit_mlp.shape)
print("X_valid_mlp:", X_valid_mlp.shape)
print("Z_fit_mlp:  ", Z_fit_mlp.shape)
print("Z_valid_mlp:", Z_valid_mlp.shape)

assert X_fit_mlp.shape == (
    len(fit_idx),
    128
)

assert X_valid_mlp.shape == (
    len(valid_idx),
    128
)

assert Z_fit_mlp.shape == (
    len(fit_idx),
    256
)

assert Z_valid_mlp.shape == (
    len(valid_idx),
    256
)


# 已经得到latent目标，可以删除较大的中间变量
del Y_fit_normalized, Y_valid_normalized
gc.collect()

print("MLP training arrays recovered successfully.")

X_fit_mlp:   (57119, 128)
X_valid_mlp: (6960, 128)
Z_fit_mlp:   (57119, 256)
Z_valid_mlp: (6960, 256)
MLP training arrays recovered successfully.


In [104]:
import gc
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers


# ============================================================
# 0. 检查所需变量
# ============================================================

required_variables = [
    "X_fit_mlp",
    "X_valid_mlp",
    "Z_fit_mlp",
    "Z_valid_mlp",
    "Y_train",
    "valid_idx",
    "z_scaler_mlp",
    "target_svd_norm256",
    "evaluate_latent_predictions"
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "缺少以下变量，请先运行MLP数据准备部分："
        + ", ".join(missing_variables)
    )

print("All required variables are available.")
print("X train:", X_fit_mlp.shape)
print("X valid:", X_valid_mlp.shape)
print("Z train:", Z_fit_mlp.shape)
print("Z valid:", Z_valid_mlp.shape)

assert X_fit_mlp.shape[1] == 128
assert X_valid_mlp.shape[1] == 128
assert Z_fit_mlp.shape[1] == 256
assert Z_valid_mlp.shape[1] == 256


# ============================================================
# 1. 尽可能保证同一种子的结果可复现
# ============================================================

try:
    tf.config.experimental.enable_op_determinism()
    print(
        "TensorFlow deterministic operations enabled."
    )
except Exception as error:
    print(
        "Could not enable full determinism:",
        error
    )


# ============================================================
# 2. 建立与原始实验完全相同的MLP
# ============================================================

def build_mlp_model(seed):
    """
    保持模型结构和训练超参数不变，
    每次只改变随机种子。
    """

    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)

    model = keras.Sequential([
        keras.Input(
            shape=(128,),
            name="atac_svd_input"
        ),

        layers.Dense(
            512,
            activation="relu",
            name="hidden_1"
        ),

        layers.Dropout(
            0.10,
            name="dropout_1"
        ),

        layers.Dense(
            256,
            activation="relu",
            name="hidden_2"
        ),

        layers.Dropout(
            0.10,
            name="dropout_2"
        ),

        layers.Dense(
            256,
            activation="linear",
            name="gex_latent_output"
        )
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss="mse"
    )

    return model


# ============================================================
# 3. 设置种子及Ridge参考分数
# ============================================================

SEEDS = [
    42,
    123,
    2026,
    7,
    2027
]

# 优先读取当前实验实际得到的Ridge分数
if "result_norm256" in globals():
    RIDGE_SCORE = float(
        result_norm256["ridge_clipped"]
    )
else:
    RIDGE_SCORE = 0.594381

print("Ridge reference score:", RIDGE_SCORE)

seed_results = []
seed_predictions = []


# ============================================================
# 4. 多种子训练与验证
# ============================================================

for seed in SEEDS:

    print("\n" + "=" * 60)
    print(f"Training MLP with seed={seed}")
    print("=" * 60)

    model = build_mlp_model(seed)

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=6,
            restore_best_weights=True,
            verbose=0
        ),

        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-5,
            verbose=0
        )
    ]

    history = model.fit(
        X_fit_mlp,
        Z_fit_mlp,
        validation_data=(
            X_valid_mlp,
            Z_valid_mlp
        ),
        epochs=50,
        batch_size=1024,
        shuffle=True,
        callbacks=callbacks,
        verbose=0
    )

    val_losses = np.asarray(
        history.history["val_loss"]
    )

    train_losses = np.asarray(
        history.history["loss"]
    )

    best_epoch_index = int(
        np.argmin(val_losses)
    )

    best_epoch = best_epoch_index + 1
    best_val_loss = float(
        val_losses[best_epoch_index]
    )
    best_train_loss = float(
        train_losses[best_epoch_index]
    )

    # 预测标准化后的Target-SVD latent
    Z_pred_scaled = model.predict(
        X_valid_mlp,
        batch_size=1024,
        verbose=0
    ).astype(np.float32)

    # 恢复到Target-SVD原始latent尺度
    Z_pred = z_scaler_mlp.inverse_transform(
        Z_pred_scaled
    ).astype(np.float32)

    assert Z_pred.shape == (
        len(valid_idx),
        256
    )

    # 在完整23418基因空间中计算Pearson分数
    raw_score, clipped_score = (
        evaluate_latent_predictions(
            Y_train[valid_idx],
            Z_pred,
            target_svd_norm256,
            batch_size=256
        )
    )

    seed_results.append({
        "seed": seed,
        "best_epoch": best_epoch,
        "best_train_loss": best_train_loss,
        "best_val_loss": best_val_loss,
        "raw_score": raw_score,
        "clipped_score": clipped_score,
        "gain_over_ridge":
            clipped_score - RIDGE_SCORE
    })

    # 保存latent预测，用于多种子集成
    seed_predictions.append(
        Z_pred.copy()
    )

    print(f"Best epoch:       {best_epoch}")
    print(
        f"Best train loss:  "
        f"{best_train_loss:.6f}"
    )
    print(
        f"Best valid loss:  "
        f"{best_val_loss:.6f}"
    )
    print(f"Raw score:        {raw_score:.6f}")
    print(
        f"Clipped score:    "
        f"{clipped_score:.6f}"
    )
    print(
        f"Gain over Ridge:  "
        f"{clipped_score - RIDGE_SCORE:+.6f}"
    )

    del model
    del history
    del Z_pred_scaled
    del Z_pred

    keras.backend.clear_session()
    gc.collect()


# ============================================================
# 5. 汇总不同种子的结果
# ============================================================

seed_results_df = pd.DataFrame(
    seed_results
)

seed_results_sorted = (
    seed_results_df
    .sort_values(
        "clipped_score",
        ascending=False
    )
    .reset_index(drop=True)
)

display(seed_results_sorted)

mean_score = float(
    seed_results_df["clipped_score"].mean()
)

std_score = float(
    seed_results_df["clipped_score"].std(
        ddof=1
    )
)

min_score = float(
    seed_results_df["clipped_score"].min()
)

max_score = float(
    seed_results_df["clipped_score"].max()
)

mean_gain = (
    mean_score - RIDGE_SCORE
)

all_above_ridge = bool(
    (
        seed_results_df["clipped_score"]
        > RIDGE_SCORE
    ).all()
)

print("\nSeed stability summary")
print("-" * 45)
print(f"Ridge score:          {RIDGE_SCORE:.6f}")
print(f"MLP mean:             {mean_score:.6f}")
print(f"MLP std:              {std_score:.6f}")
print(f"MLP minimum:          {min_score:.6f}")
print(f"MLP maximum:          {max_score:.6f}")
print(f"MLP range:            {max_score - min_score:.6f}")
print(f"Mean gain over Ridge: {mean_gain:+.6f}")
print(f"All seeds beat Ridge: {all_above_ridge}")


# ============================================================
# 6. 多种子MLP latent预测平均
# ============================================================

Z_mlp_seed_ensemble = np.mean(
    np.stack(
        seed_predictions,
        axis=0
    ),
    axis=0
).astype(np.float32)

assert Z_mlp_seed_ensemble.shape == (
    len(valid_idx),
    256
)

ensemble_raw, ensemble_clipped = (
    evaluate_latent_predictions(
        Y_train[valid_idx],
        Z_mlp_seed_ensemble,
        target_svd_norm256,
        batch_size=256
    )
)

best_single_score = float(
    seed_results_df[
        "clipped_score"
    ].max()
)

print("\nMulti-seed MLP ensemble")
print("-" * 45)
print(
    f"Raw score:               "
    f"{ensemble_raw:.6f}"
)
print(
    f"Clipped score:           "
    f"{ensemble_clipped:.6f}"
)
print(
    f"Gain over Ridge:         "
    f"{ensemble_clipped - RIDGE_SCORE:+.6f}"
)
print(
    f"Gain over mean MLP:      "
    f"{ensemble_clipped - mean_score:+.6f}"
)
print(
    f"Gain over best single:   "
    f"{ensemble_clipped - best_single_score:+.6f}"
)


# ============================================================
# 7. 保存实验结果表
# ============================================================

seed_results_df.to_csv(
    WORK_DIR / "mlp_seed_stability_results.csv",
    index=False
)

print(
    "\nSaved:",
    WORK_DIR / "mlp_seed_stability_results.csv"
)

All required variables are available.
X train: (57119, 128)
X valid: (6960, 128)
Z train: (57119, 256)
Z valid: (6960, 256)
TensorFlow deterministic operations enabled.
Ridge reference score: 0.5943810939788818

Training MLP with seed=42


2026-10-03 07:30:14.016673: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-10-03 07:31:04.668188: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Best epoch:       23
Best train loss:  0.911692
Best valid loss:  0.875673
Raw score:        0.597296
Clipped score:    0.597377
Gain over Ridge:  +0.002996

Training MLP with seed=123


2026-10-03 07:31:19.721068: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-10-03 07:32:11.124803: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Best epoch:       24
Best train loss:  0.913006
Best valid loss:  0.884600
Raw score:        0.596286
Clipped score:    0.596385
Gain over Ridge:  +0.002004

Training MLP with seed=2026


...

Best epoch:       20
Best train loss:  0.916400
Best valid loss:  0.878300
Raw score:        0.596590
Clipped score:    0.596656
Gain over Ridge:  +0.002275

Training MLP with seed=2027


2026-10-03 07:34:19.371667: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-10-03 07:35:06.249269: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Best epoch:       21
Best train loss:  0.914357
Best valid loss:  0.879117
Raw score:        0.596710
Clipped score:    0.596761
Gain over Ridge:  +0.002380


,seed,best_epoch,best_train_loss,best_val_loss,raw_score,clipped_score,gain_over_ridge
0,42,23,0.911692,0.875673,0.597296,0.597377,0.002996
1,2026,14,0.923980,0.880134,0.596701,0.596762,0.002381
2,2027,21,0.914357,0.879117,0.596710,0.596761,0.002380
3,7,20,0.916400,0.878300,0.596590,0.596656,0.002275
4,123,24,0.913006,0.884600,0.596286,0.596385,0.002004



Seed stability summary
---------------------------------------------
Ridge score:          0.594381
MLP mean:             0.596788
MLP std:              0.000363
MLP minimum:          0.596385
MLP maximum:          0.597377
MLP range:            0.000992
Mean gain over Ridge: +0.002407
All seeds beat Ridge: True

Multi-seed MLP ensemble
---------------------------------------------
Raw score:               0.597418
Clipped score:           0.597468
Gain over Ridge:         +0.003087
Gain over mean MLP:      +0.000680
Gain over best single:   +0.000091

Saved: /kaggle/working/multiome_sparse/mlp_seed_stability_results.csv


## Train the model on full data

In [105]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path(
    "/kaggle/input/competitions/"
    "open-problems-multimodal"
)

evaluation_head = pd.read_csv(
    DATA_DIR / "evaluation_ids.csv",
    nrows=10
)

submission_head = pd.read_csv(
    DATA_DIR / "sample_submission.csv",
    nrows=10
)

print("evaluation_ids columns:")
print(evaluation_head.columns.tolist())
display(evaluation_head)

print("\nsample_submission columns:")
print(submission_head.columns.tolist())
display(submission_head)

print("\nAvailable model variables:")
for variable_name in [
    "X_train_svd",
    "X_test_svd",
    "Y_train",
    "train_input_cells",
    "test_input_cells",
    "train_target_genes"
]:
    if variable_name in globals():
        value = globals()[variable_name]
        print(
            f"{variable_name:25s}",
            "OK",
            getattr(value, "shape", "")
        )
    else:
        print(
            f"{variable_name:25s}",
            "MISSING"
        )

print("\nSaved SVD arrays:")
for filename in [
    "X_train_multi_svd128.npy",
    "X_test_multi_svd128.npy",
    "input_multi_svd128.joblib"
]:
    path = WORK_DIR / filename
    print(
        filename,
        "exists:",
        path.exists(),
        "size:",
        (
            f"{path.stat().st_size / 1024**2:.1f} MB"
            if path.exists()
            else "N/A"
        )
    )

evaluation_ids columns:
['row_id', 'cell_id', 'gene_id']


,row_id,cell_id,gene_id
0,0,c2150f55becb,CD86
1,1,c2150f55becb,CD274
2,2,c2150f55becb,CD270
3,3,c2150f55becb,CD155
4,4,c2150f55becb,CD112
5,5,c2150f55becb,CD47
6,6,c2150f55becb,CD48
7,7,c2150f55becb,CD40
8,8,c2150f55becb,CD154
9,9,c2150f55becb,CD52



sample_submission columns:
['row_id', 'target']


,row_id,target
0,0,0.0
1,1,0.0
2,2,0.0
3,3,0.0
4,4,0.0
5,5,0.0
6,6,0.0
7,7,0.0
8,8,0.0
9,9,0.0



Available model variables:
X_train_svd               OK (105942, 128)
X_test_svd                MISSING
Y_train                   OK (105942, 23418)
train_input_cells         OK (105942,)
test_input_cells          OK (55935,)
train_target_genes        MISSING

Saved SVD arrays:
X_train_multi_svd128.npy exists: False size: N/A
X_test_multi_svd128.npy exists: False size: N/A
input_multi_svd128.joblib exists: False size: N/A


In [107]:
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# 1. 搜索所有可能的SVD文件
# ============================================================

SEARCH_ROOTS = [
    Path("/kaggle/working"),
    Path("/kaggle/input")
]

train_candidates = []
test_candidates = []
svd_model_candidates = []

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    train_candidates.extend(
        root.rglob("X_train_multi_svd128.npy")
    )

    test_candidates.extend(
        root.rglob("X_test_multi_svd128.npy")
    )

    svd_model_candidates.extend(
        root.rglob("input_multi_svd128.joblib")
    )

print("Train SVD candidates:")
for path in train_candidates:
    print(" ", path)

print("\nTest SVD candidates:")
for path in test_candidates:
    print(" ", path)

print("\nInput-SVD model candidates:")
for path in svd_model_candidates:
    print(" ", path)


# ============================================================
# 2. 找到与当前X_train_svd完全匹配的文件
# ============================================================

assert X_train_svd.shape == (105942, 128)

# 抽取多个位置进行匹配，避免只比较第一行
check_rows = np.array([
    0,
    1,
    100,
    1000,
    10000,
    50000,
    100000,
    105941
])

matching_pairs = []

for train_path in train_candidates:

    try:
        candidate_train = np.load(
            train_path,
            mmap_mode="r"
        )

        if candidate_train.shape != (105942, 128):
            continue

        is_match = np.allclose(
            candidate_train[check_rows],
            X_train_svd[check_rows],
            rtol=1e-6,
            atol=1e-7
        )

        if not is_match:
            continue

        sibling_test_path = (
            train_path.parent
            / "X_test_multi_svd128.npy"
        )

        if not sibling_test_path.exists():
            print(
                "Matching train file found, "
                "but sibling test file missing:",
                train_path
            )
            continue

        candidate_test = np.load(
            sibling_test_path,
            mmap_mode="r"
        )

        if candidate_test.shape != (55935, 128):
            print(
                "Wrong test shape:",
                sibling_test_path,
                candidate_test.shape
            )
            continue

        matching_pairs.append({
            "train_path": train_path,
            "test_path": sibling_test_path
        })

    except Exception as error:
        print(
            "Could not inspect:",
            train_path,
            error
        )


print("\nMatching artifact pairs:", len(matching_pairs))

for pair in matching_pairs:
    print("\nTrain:", pair["train_path"])
    print("Test: ", pair["test_path"])


if len(matching_pairs) == 0:
    raise FileNotFoundError(
        "没有找到与当前X_train_svd匹配的测试SVD文件。"
    )


# 如果存在多个相同备份，优先使用第一个
selected_pair = matching_pairs[0]

X_TEST_SVD_PATH = selected_pair["test_path"]

X_test_svd = np.load(
    X_TEST_SVD_PATH,
    mmap_mode="r"
)

assert X_test_svd.shape == (55935, 128)
assert X_test_svd.dtype == np.float32

print("\nSelected test SVD:")
print(X_TEST_SVD_PATH)
print("X_test_svd:", X_test_svd.shape)
print("dtype:", X_test_svd.dtype)

Train SVD candidates:
  /kaggle/input/datasets/huangyufeid/multiome-svd128-backup/X_train_multi_svd128.npy

Test SVD candidates:
  /kaggle/input/datasets/huangyufeid/multiome-svd128-backup/X_test_multi_svd128.npy

Input-SVD model candidates:

Matching artifact pairs: 1

Train: /kaggle/input/datasets/huangyufeid/multiome-svd128-backup/X_train_multi_svd128.npy
Test:  /kaggle/input/datasets/huangyufeid/multiome-svd128-backup/X_test_multi_svd128.npy

Selected test SVD:
/kaggle/input/datasets/huangyufeid/multiome-svd128-backup/X_test_multi_svd128.npy
X_test_svd: (55935, 128)
dtype: float32


In [108]:
# ============================================================
# 0. Imports and paths
# ============================================================

from pathlib import Path
import gc
import time
import random
import joblib

import numpy as np
import pandas as pd
import scipy.sparse as sp
import tensorflow as tf

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, normalize

from tensorflow import keras
from tensorflow.keras import layers


DATA_DIR = Path(
    "/kaggle/input/competitions/"
    "open-problems-multimodal"
)

SVD_BACKUP_DIR = Path(
    "/kaggle/input/datasets/huangyufeid/"
    "multiome-svd128-backup"
)

FINAL_DIR = Path(
    "/kaggle/working/multiome_final"
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

X_TRAIN_SVD_PATH = (
    SVD_BACKUP_DIR
    / "X_train_multi_svd128.npy"
)

X_TEST_SVD_PATH = (
    SVD_BACKUP_DIR
    / "X_test_multi_svd128.npy"
)

EVALUATION_IDS_PATH = (
    DATA_DIR
    / "evaluation_ids.csv"
)

MULTI_TARGET_PATH = (
    DATA_DIR
    / "train_multi_targets.h5"
)

print("Final artifact directory:", FINAL_DIR)

Final artifact directory: /kaggle/working/multiome_final


In [109]:
# ============================================================
# 1. Load matched input-SVD arrays
# ============================================================

X_train_svd_final = np.load(
    X_TRAIN_SVD_PATH,
    mmap_mode="r"
)

X_test_svd_final = np.load(
    X_TEST_SVD_PATH,
    mmap_mode="r"
)

assert X_train_svd_final.shape == (105942, 128)
assert X_test_svd_final.shape == (55935, 128)

assert X_train_svd_final.dtype == np.float32
assert X_test_svd_final.dtype == np.float32

assert Y_train.shape == (105942, 23418)
assert sp.issparse(Y_train)

assert len(train_input_cells) == 105942
assert len(test_input_cells) == 55935

print("X train:", X_train_svd_final.shape)
print("X test: ", X_test_svd_final.shape)
print("Y train:", Y_train.shape)

X train: (105942, 128)
X test:  (55935, 128)
Y train: (105942, 23418)


In [110]:
# ============================================================
# 2. Recover target gene names
# ============================================================

target_sample = pd.read_hdf(
    MULTI_TARGET_PATH,
    key="train_multi_targets",
    start=0,
    stop=1
)

train_target_genes = (
    target_sample.columns
    .astype(str)
    .to_numpy()
)

del target_sample
gc.collect()

assert train_target_genes.shape == (23418,)
assert len(np.unique(train_target_genes)) == 23418

test_input_cells_final = np.asarray(
    test_input_cells
).astype(str)

assert len(np.unique(test_input_cells_final)) == 55935

print("Target genes:", train_target_genes.shape)
print("First genes:", train_target_genes[:5])
print("Test cells:", test_input_cells_final.shape)

Target genes: (23418,)
First genes: ['ENSG00000121410' 'ENSG00000268895' 'ENSG00000175899' 'ENSG00000245105'
 'ENSG00000166535']
Test cells: (55935,)


In [111]:
import shutil

disk_usage = shutil.disk_usage("/kaggle/working")

print(
    "Free disk space:",
    f"{disk_usage.free / 1024**3:.2f} GB"
)

Free disk space: 13.86 GB


In [112]:
# ============================================================
# 3. Fit final Target-SVD256 using all training cells
# ============================================================

N_TARGET_COMPONENTS = 256

print("Normalizing all training targets...")

start_time = time.time()

Y_train_normalized_final = normalize(
    Y_train,
    norm="l2",
    axis=1,
    copy=True
).astype(np.float32)

print(
    "Normalized target shape:",
    Y_train_normalized_final.shape
)

target_svd_final = TruncatedSVD(
    n_components=N_TARGET_COMPONENTS,
    algorithm="randomized",
    n_iter=5,
    random_state=42
)

print("Fitting final Target-SVD256...")

Z_train_final = target_svd_final.fit_transform(
    Y_train_normalized_final
).astype(np.float32)

elapsed = (time.time() - start_time) / 60

print("Target latent shape:", Z_train_final.shape)
print(f"Target-SVD time: {elapsed:.1f} minutes")
print(
    "Explained variance:",
    target_svd_final
    .explained_variance_ratio_
    .sum()
)

assert Z_train_final.shape == (105942, 256)

del Y_train_normalized_final
gc.collect()

Normalizing all training targets...
Normalized target shape: (105942, 23418)
Fitting final Target-SVD256...
Target latent shape: (105942, 256)
Target-SVD time: 14.4 minutes
Explained variance: 0.12163314


0

In [113]:
# ============================================================
# 4. Fit final scalers
# ============================================================

print("Scaling input latent...")

x_scaler_final = StandardScaler()

X_train_scaled_final = (
    x_scaler_final
    .fit_transform(X_train_svd_final)
    .astype(np.float32)
)

X_test_scaled_final = (
    x_scaler_final
    .transform(X_test_svd_final)
    .astype(np.float32)
)

print("Scaling target latent...")

z_scaler_final = StandardScaler()

Z_train_scaled_final = (
    z_scaler_final
    .fit_transform(Z_train_final)
    .astype(np.float32)
)

assert X_train_scaled_final.shape == (105942, 128)
assert X_test_scaled_final.shape == (55935, 128)
assert Z_train_scaled_final.shape == (105942, 256)

assert np.isfinite(X_train_scaled_final).all()
assert np.isfinite(X_test_scaled_final).all()
assert np.isfinite(Z_train_scaled_final).all()

print("X train scaled:", X_train_scaled_final.shape)
print("X test scaled: ", X_test_scaled_final.shape)
print("Z train scaled:", Z_train_scaled_final.shape)

del Z_train_final
gc.collect()

Scaling input latent...
Scaling target latent...
X train scaled: (105942, 128)
X test scaled:  (55935, 128)
Z train scaled: (105942, 256)


0

In [114]:
# ============================================================
# 5. Final MLP definition
# ============================================================

def build_final_mlp(seed):

    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)

    inputs = keras.Input(
        shape=(128,),
        name="atac_svd_input"
    )

    x = layers.Dense(
        256,
        activation="swish"
    )(inputs)

    x = layers.Dropout(
        0.10,
        seed=seed
    )(x)

    x = layers.Dense(
        256,
        activation="swish"
    )(x)

    x = layers.Dropout(
        0.10,
        seed=seed + 1
    )(x)

    outputs = layers.Dense(
        256,
        name="gex_latent_output"
    )(x)

    model = keras.Model(
        inputs=inputs,
        outputs=outputs
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss="mse"
    )

    return model

In [118]:
def final_lr_schedule(epoch, current_lr):
    """
    epoch是从0开始：
    1–15:  1e-3
    16–18: 5e-4
    19–21: 2.5e-4
    """

    if epoch < 15:
        return 1e-3
    elif epoch < 18:
        return 5e-4
    else:
        return 2.5e-4

In [120]:
# ============================================================
# Final training configuration
# 必须在五种子训练循环之前运行
# ============================================================

FINAL_SEEDS = [
    42,
    123,
    2026,
    7,
    2027
]

FINAL_EPOCHS = 21
FINAL_BATCH_SIZE = 1024


def final_lr_schedule(epoch, current_lr):
    """
    epoch从0开始：

    Epoch 1–15:  1e-3
    Epoch 16–18: 5e-4
    Epoch 19–21: 2.5e-4
    """

    if epoch < 15:
        return 1e-3
    elif epoch < 18:
        return 5e-4
    else:
        return 2.5e-4


print("Final seeds:", FINAL_SEEDS)
print("Final epochs:", FINAL_EPOCHS)
print("Batch size:", FINAL_BATCH_SIZE)

assert len(FINAL_SEEDS) == 5
assert FINAL_EPOCHS == 21

Final seeds: [42, 123, 2026, 7, 2027]
Final epochs: 21
Batch size: 1024


In [121]:
# ============================================================
# 6. Train five final MLP models
# ============================================================

try:
    tf.config.experimental.enable_op_determinism()
    print("Deterministic TensorFlow operations enabled.")
except Exception as error:
    print("Full determinism unavailable:", error)


Z_test_scaled_sum = np.zeros(
    (55935, 256),
    dtype=np.float32
)

all_training_histories = []

for seed in FINAL_SEEDS:

    print("\n" + "=" * 70)
    print(f"Training final MLP: seed={seed}")
    print("=" * 70)

    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

    model = build_final_mlp(seed)

    lr_callback = keras.callbacks.LearningRateScheduler(
        final_lr_schedule,
        verbose=1
    )

    start_time = time.time()

    history = model.fit(
        X_train_scaled_final,
        Z_train_scaled_final,
        epochs=FINAL_EPOCHS,
        batch_size=FINAL_BATCH_SIZE,
        shuffle=True,
        callbacks=[lr_callback],
        verbose=2
    )

    elapsed = (time.time() - start_time) / 60

    print(
        f"Training completed in "
        f"{elapsed:.2f} minutes"
    )

    # 保存训练历史
    history_df = pd.DataFrame(
        history.history
    )

    history_df["epoch"] = (
        np.arange(len(history_df)) + 1
    )

    history_df["seed"] = seed

    all_training_histories.append(
        history_df
    )

    # 测试latent预测
    Z_test_seed_scaled = model.predict(
        X_test_scaled_final,
        batch_size=1024,
        verbose=1
    ).astype(np.float32)

    assert Z_test_seed_scaled.shape == (55935, 256)
    assert np.isfinite(Z_test_seed_scaled).all()

    Z_test_scaled_sum += Z_test_seed_scaled

    # 保存当前模型
    model_path = (
        FINAL_DIR
        / f"final_mlp_seed_{seed}.keras"
    )

    model.save(model_path)

    print("Saved:", model_path)

    del model
    del history
    del history_df
    del Z_test_seed_scaled

    keras.backend.clear_session()
    gc.collect()

Deterministic TensorFlow operations enabled.

Training final MLP: seed=42

Epoch 1: LearningRateScheduler setting learning rate to 0.001.
Epoch 1/21


2026-10-03 08:16:59.956857: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


104/104 - 5s - 44ms/step - loss: 0.9778 - learning_rate: 0.0010

Epoch 2: LearningRateScheduler setting learning rate to 0.001.
Epoch 2/21
104/104 - 2s - 23ms/step - loss: 0.9358 - learning_rate: 0.0010

Epoch 3: LearningRateScheduler setting learning rate to 0.001.
Epoch 3/21
104/104 - 2s - 23ms/step - loss: 0.9267 - learning_rate: 0.0010

Epoch 4: LearningRateScheduler setting learning rate to 0.001.
Epoch 4/21
104/104 - 2s - 23ms/step - loss: 0.9200 - learning_rate: 0.0010

Epoch 5: LearningRateScheduler setting learning rate to 0.001.
Epoch 5/21
104/104 - 2s - 23ms/step - loss: 0.9152 - learning_rate: 0.0010

Epoch 6: LearningRateScheduler setting learning rate to 0.001.
Epoch 6/21
104/104 - 2s - 22ms/step - loss: 0.9116 - learning_rate: 0.0010

Epoch 7: LearningRateScheduler setting learning rate to 0.001.
Epoch 7/21
104/104 - 2s - 22ms/step - loss: 0.9089 - learning_rate: 0.0010

Epoch 8: LearningRateScheduler setting learning rate to 0.001.
Epoch 8/21
104/104 - 2s - 22ms/step - 

2026-10-03 08:17:51.297629: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
Saved: /kaggle/working/multiome_final/final_mlp_seed_42.keras

Training final MLP: seed=123

Epoch 1: LearningRateScheduler setting learning rate to 0.001.
Epoch 1/21
104/104 - 4s - 39ms/step - loss: 0.9779 - learning_rate: 0.0010

Epoch 2: LearningRateScheduler setting learning rate to 0.001.
Epoch 2/21
104/104 - 2s - 22ms/step - loss: 0.9360 - learning_rate: 0.0010

Epoch 3: LearningRateScheduler setting learning rate to 0.001.
Epoch 3/21
104/104 - 2s - 23ms/step - loss: 0.9269 - learning_rate: 0.0010

Epoch 4: LearningRateScheduler setting learning rate to 0.001.
Epoch 4/21
104/104 - 2s - 23ms/step - loss: 0.9202 - learning_rate: 0.0010

Epoch 5: LearningRateScheduler setting learning rate to 0.001.
Epoch 5/21
104/104 - 2s - 22ms/step - loss: 0.9153 - learning_rate: 0.0010

Epoch 6: LearningRateScheduler setting learning rate to 0.001.
Epoch 6/21
104/104 - 2s - 22ms/step - loss: 0.9117 - learning_rate: 0.0010

Epoch 7: LearningRateScheduler set

...

55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
Saved: /kaggle/working/multiome_final/final_mlp_seed_2026.keras

Training final MLP: seed=7

Epoch 1: LearningRateScheduler setting learning rate to 0.001.
Epoch 1/21
104/104 - 4s - 37ms/step - loss: 0.9785 - learning_rate: 0.0010

Epoch 2: LearningRateScheduler setting learning rate to 0.001.
Epoch 2/21
104/104 - 2s - 22ms/step - loss: 0.9359 - learning_rate: 0.0010

Epoch 3: LearningRateScheduler setting learning rate to 0.001.
Epoch 3/21
104/104 - 2s - 22ms/step - loss: 0.9267 - learning_rate: 0.0010

Epoch 4: LearningRateScheduler setting learning rate to 0.001.
Epoch 4/21
104/104 - 2s - 22ms/step - loss: 0.9200 - learning_rate: 0.0010

Epoch 5: LearningRateScheduler setting learning rate to 0.001.
Epoch 5/21
104/104 - 2s - 22ms/step - loss: 0.9151 - learning_rate: 0.0010

Epoch 6: LearningRateScheduler setting learning rate to 0.001.
Epoch 6/21
104/104 - 2s - 22ms/step - loss: 0.9116 - learning_rate: 0.0010

Epoch 7: LearningRateScheduler set

2026-10-03 08:20:29.714938: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
Saved: /kaggle/working/multiome_final/final_mlp_seed_7.keras

Training final MLP: seed=2027

Epoch 1: LearningRateScheduler setting learning rate to 0.001.
Epoch 1/21
104/104 - 4s - 38ms/step - loss: 0.9780 - learning_rate: 0.0010

Epoch 2: LearningRateScheduler setting learning rate to 0.001.
Epoch 2/21
104/104 - 2s - 22ms/step - loss: 0.9359 - learning_rate: 0.0010

Epoch 3: LearningRateScheduler setting learning rate to 0.001.
Epoch 3/21
104/104 - 2s - 23ms/step - loss: 0.9269 - learning_rate: 0.0010

Epoch 4: LearningRateScheduler setting learning rate to 0.001.
Epoch 4/21
104/104 - 2s - 22ms/step - loss: 0.9203 - learning_rate: 0.0010

Epoch 5: LearningRateScheduler setting learning rate to 0.001.
Epoch 5/21
104/104 - 2s - 22ms/step - loss: 0.9152 - learning_rate: 0.0010

Epoch 6: LearningRateScheduler setting learning rate to 0.001.
Epoch 6/21
104/104 - 2s - 22ms/step - loss: 0.9117 - learning_rate: 0.0010

Epoch 7: LearningRateScheduler se

2026-10-03 08:21:22.346329: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
Saved: /kaggle/working/multiome_final/final_mlp_seed_2027.keras


In [122]:
# ============================================================
# 7. Average five MLP predictions
# ============================================================

Z_test_scaled_ensemble = (
    Z_test_scaled_sum
    / len(FINAL_SEEDS)
).astype(np.float32)

# 从标准化latent恢复到Target-SVD原始latent
Z_test_latent_final = (
    z_scaler_final
    .inverse_transform(Z_test_scaled_ensemble)
    .astype(np.float32)
)

assert Z_test_latent_final.shape == (55935, 256)
assert np.isfinite(Z_test_latent_final).all()

print(
    "Final test latent:",
    Z_test_latent_final.shape
)

del Z_test_scaled_sum
del Z_test_scaled_ensemble
gc.collect()

Final test latent: (55935, 256)


0

save

In [124]:
# ============================================================
# Fix string-array dtype and continue saving
# 不需要重新训练MLP
# ============================================================

# 转换成固定长度Unicode，而不是object
train_target_genes_save = np.asarray(
    train_target_genes,
    dtype=str
)

test_input_cells_save = np.asarray(
    test_input_cells_final,
    dtype=str
)

print(
    "Gene dtype:",
    train_target_genes_save.dtype
)

print(
    "Cell dtype:",
    test_input_cells_save.dtype
)

assert not train_target_genes_save.dtype.hasobject
assert not test_input_cells_save.dtype.hasobject

assert train_target_genes_save.shape == (23418,)
assert test_input_cells_save.shape == (55935,)


# 重新保存；会覆盖可能留下的不完整文件
np.save(
    FINAL_DIR / "target_gene_ids.npy",
    train_target_genes_save,
    allow_pickle=False
)

np.save(
    FINAL_DIR / "test_multi_cell_ids.npy",
    test_input_cells_save,
    allow_pickle=False
)


# 保存训练历史
training_history_final = pd.concat(
    all_training_histories,
    ignore_index=True
)

training_history_final.to_csv(
    FINAL_DIR / "final_mlp_training_history.csv",
    index=False
)

print("String ID arrays and training history saved.")

Gene dtype: <U15
Cell dtype: <U12
String ID arrays and training history saved.


In [125]:
# ============================================================
# Verify saved artifacts
# ============================================================

saved_genes = np.load(
    FINAL_DIR / "target_gene_ids.npy",
    allow_pickle=False
)

saved_cells = np.load(
    FINAL_DIR / "test_multi_cell_ids.npy",
    allow_pickle=False
)

saved_latent = np.load(
    FINAL_DIR / "Z_test_latent_ensemble.npy",
    mmap_mode="r"
)

assert np.array_equal(
    saved_genes,
    train_target_genes_save
)

assert np.array_equal(
    saved_cells,
    test_input_cells_save
)

assert saved_latent.shape == (55935, 256)
assert np.isfinite(saved_latent).all()

required_files = [
    "target_svd256_final.joblib",
    "x_scaler_final.joblib",
    "z_scaler_final.joblib",
    "Z_test_latent_ensemble.npy",
    "target_gene_ids.npy",
    "test_multi_cell_ids.npy",
    "final_mlp_training_history.csv",
    "final_mlp_seed_42.keras",
    "final_mlp_seed_123.keras",
    "final_mlp_seed_2026.keras",
    "final_mlp_seed_7.keras",
    "final_mlp_seed_2027.keras"
]

print("\nFinal artifact check")
print("-" * 60)

for filename in required_files:
    path = FINAL_DIR / filename

    print(
        f"{filename:40s}",
        "OK" if path.exists() else "MISSING",
        (
            f"{path.stat().st_size / 1024**2:.2f} MB"
            if path.exists()
            else ""
        )
    )

assert all(
    (FINAL_DIR / filename).exists()
    for filename in required_files
)

print("\nAll final artifacts are valid.")


Final artifact check
------------------------------------------------------------
target_svd256_final.joblib               OK 22.87 MB
x_scaler_final.joblib                    OK 0.00 MB
z_scaler_final.joblib                    OK 0.01 MB
Z_test_latent_ensemble.npy               OK 54.62 MB
target_gene_ids.npy                      OK 1.34 MB
test_multi_cell_ids.npy                  OK 2.56 MB
final_mlp_training_history.csv           OK 0.00 MB
final_mlp_seed_42.keras                  OK 1.91 MB
final_mlp_seed_123.keras                 OK 1.91 MB
final_mlp_seed_2026.keras                OK 1.91 MB
final_mlp_seed_7.keras                   OK 1.91 MB
final_mlp_seed_2027.keras                OK 1.91 MB

All final artifacts are valid.


In [126]:
# ============================================================
# 9. Prepare evaluation ID mappings
# ============================================================

cell_to_index = {
    cell_id: index
    for index, cell_id
    in enumerate(test_input_cells_final)
}

gene_to_index = {
    gene_id: index
    for index, gene_id
    in enumerate(train_target_genes)
}

assert len(cell_to_index) == 55935
assert len(gene_to_index) == 23418

target_components = np.asarray(
    target_svd_final.components_,
    dtype=np.float32
)

assert target_components.shape == (256, 23418)

MULTIOME_PART_PATH = (
    FINAL_DIR
    / "multiome_submission_part.csv"
)

# 删除之前可能存在的不完整文件
MULTIOME_PART_PATH.unlink(
    missing_ok=True
)

print("Cell mapping:", len(cell_to_index))
print("Gene mapping:", len(gene_to_index))
print("Output:", MULTIOME_PART_PATH)

Cell mapping: 55935
Gene mapping: 23418
Output: /kaggle/working/multiome_final/multiome_submission_part.csv


In [127]:
# ============================================================
# 10. Generate Multiome submission part in chunks
# ============================================================

CSV_CHUNK_SIZE = 250_000
PAIR_BATCH_SIZE = 20_000

first_write = True
total_evaluation_rows = 0
total_multiome_rows = 0

start_time = time.time()

evaluation_reader = pd.read_csv(
    EVALUATION_IDS_PATH,
    usecols=[
        "row_id",
        "cell_id",
        "gene_id"
    ],
    chunksize=CSV_CHUNK_SIZE
)

for chunk_number, evaluation_chunk in enumerate(
    evaluation_reader,
    start=1
):

    total_evaluation_rows += len(
        evaluation_chunk
    )

    # 只有Multiome test cell会出现在cell_to_index中
    mapped_cells = evaluation_chunk[
        "cell_id"
    ].map(cell_to_index)

    is_multiome = mapped_cells.notna()

    if not is_multiome.any():
        if chunk_number % 20 == 0:
            print(
                f"Chunk {chunk_number}: "
                "no Multiome rows yet"
            )
        continue

    multi_chunk = evaluation_chunk.loc[
        is_multiome,
        ["row_id", "gene_id"]
    ].copy()

    cell_indices = (
        mapped_cells.loc[is_multiome]
        .to_numpy(dtype=np.int32)
    )

    gene_indices_series = (
        multi_chunk["gene_id"]
        .map(gene_to_index)
    )

    if gene_indices_series.isna().any():
        missing_genes = (
            multi_chunk.loc[
                gene_indices_series.isna(),
                "gene_id"
            ]
            .drop_duplicates()
            .head(20)
            .tolist()
        )

        raise KeyError(
            "evaluation_ids中出现了未知Multiome基因："
            + str(missing_genes)
        )

    gene_indices = (
        gene_indices_series
        .to_numpy(dtype=np.int32)
    )

    row_ids = multi_chunk[
        "row_id"
    ].to_numpy(dtype=np.int64)

    predictions = np.empty(
        len(multi_chunk),
        dtype=np.float32
    )

    # 更小的pair batch避免产生过大的临时数组
    for pair_start in range(
        0,
        len(multi_chunk),
        PAIR_BATCH_SIZE
    ):

        pair_stop = min(
            pair_start + PAIR_BATCH_SIZE,
            len(multi_chunk)
        )

        batch_cell_indices = cell_indices[
            pair_start:pair_stop
        ]

        batch_gene_indices = gene_indices[
            pair_start:pair_stop
        ]

        latent_rows = Z_test_latent_final[
            batch_cell_indices
        ]

        component_rows = (
            target_components[
                :,
                batch_gene_indices
            ]
            .T
        )

        batch_predictions = np.einsum(
            "ij,ij->i",
            latent_rows,
            component_rows,
            optimize=True
        ).astype(np.float32)

        # 与验证阶段完全相同：GEX预测不能为负
        np.maximum(
            batch_predictions,
            0,
            out=batch_predictions
        )

        predictions[
            pair_start:pair_stop
        ] = batch_predictions

        del latent_rows
        del component_rows
        del batch_predictions

    output_chunk = pd.DataFrame({
        "row_id": row_ids,
        "target": predictions
    })

    output_chunk.to_csv(
        MULTIOME_PART_PATH,
        mode="w" if first_write else "a",
        header=first_write,
        index=False,
        float_format="%.8g"
    )

    first_write = False
    total_multiome_rows += len(output_chunk)

    if chunk_number % 10 == 0:
        elapsed_minutes = (
            time.time() - start_time
        ) / 60

        print(
            f"Chunk {chunk_number} | "
            f"scanned={total_evaluation_rows:,} | "
            f"Multiome={total_multiome_rows:,} | "
            f"time={elapsed_minutes:.1f} min"
        )

    del evaluation_chunk
    del mapped_cells
    del multi_chunk
    del cell_indices
    del gene_indices_series
    del gene_indices
    del row_ids
    del predictions
    del output_chunk

    gc.collect()

Chunk 20: no Multiome rows yet
Chunk 30 | scanned=7,500,000 | Multiome=687,180 | time=0.2 min
Chunk 40 | scanned=10,000,000 | Multiome=3,187,180 | time=0.5 min
Chunk 50 | scanned=12,500,000 | Multiome=5,687,180 | time=0.7 min
Chunk 60 | scanned=15,000,000 | Multiome=8,187,180 | time=1.0 min
Chunk 70 | scanned=17,500,000 | Multiome=10,687,180 | time=1.2 min
Chunk 80 | scanned=20,000,000 | Multiome=13,187,180 | time=1.5 min
Chunk 90 | scanned=22,500,000 | Multiome=15,687,180 | time=1.7 min
Chunk 100 | scanned=25,000,000 | Multiome=18,187,180 | time=2.0 min
Chunk 110 | scanned=27,500,000 | Multiome=20,687,180 | time=2.2 min
Chunk 120 | scanned=30,000,000 | Multiome=23,187,180 | time=2.5 min
Chunk 130 | scanned=32,500,000 | Multiome=25,687,180 | time=2.7 min
Chunk 140 | scanned=35,000,000 | Multiome=28,187,180 | time=2.9 min
Chunk 150 | scanned=37,500,000 | Multiome=30,687,180 | time=3.2 min
Chunk 160 | scanned=40,000,000 | Multiome=33,187,180 | time=3.4 min
Chunk 170 | scanned=42,500,000 

In [128]:
# ============================================================
# 11. Validate Multiome submission part
# ============================================================

assert MULTIOME_PART_PATH.exists()
assert total_multiome_rows > 0

multiome_part_head = pd.read_csv(
    MULTIOME_PART_PATH,
    nrows=10
)

display(multiome_part_head)

print("\nGeneration completed")
print("-" * 50)
print(
    "Evaluation rows scanned:",
    f"{total_evaluation_rows:,}"
)
print(
    "Multiome rows written:",
    f"{total_multiome_rows:,}"
)
print(
    "Output size:",
    f"{MULTIOME_PART_PATH.stat().st_size / 1024**3:.2f} GB"
)
print("Output:", MULTIOME_PART_PATH)

assert list(multiome_part_head.columns) == [
    "row_id",
    "target"
]

assert np.isfinite(
    multiome_part_head["target"]
).all()

assert (
    multiome_part_head["target"] >= 0
).all()

print("Basic submission checks passed.")

,row_id,target
0,6812820,0.000008
1,6812821,0.032102
2,6812822,0.002943
3,6812823,0.003204
4,6812824,0.000012
5,6812825,0.000858
6,6812826,0.000049
7,6812827,0.000708
8,6812828,0.000009
9,6812829,0.000059



Generation completed
--------------------------------------------------
Evaluation rows scanned: 65,744,180
Multiome rows written: 58,931,360
Output size: 1.17 GB
Output: /kaggle/working/multiome_final/multiome_submission_part.csv
Basic submission checks passed.


## ============================================                       
## Save outputs

In [ ]:
!du -sh /kaggle/working
!du -ah /kaggle/working | sort -rh | head -25

In [138]:
from pathlib import Path
import shutil
import tempfile

WORK_DIR = Path("/kaggle/working/multiome_sparse")

paths_to_move = [
    WORK_DIR / "train_multi_inputs_matrix.npz",
    WORK_DIR / "test_multi_inputs_matrix.npz",
    WORK_DIR / "train_multi_targets_matrix.npz",
]

for path in paths_to_move:
    if not path.is_file():
        raise FileNotFoundError(path)

backup_zip = Path("/kaggle/working/multiome_artifacts_backup.zip")
if backup_zip.is_file():
    paths_to_move.append(backup_zip)

Path("/kaggle/temp").mkdir(parents=True, exist_ok=True)
temporary_dir = Path(tempfile.mkdtemp(
    prefix="multiome_save_",
    dir="/kaggle/temp"
))

moved_pairs = []

for source in paths_to_move:
    destination = temporary_dir / source.name
    shutil.move(str(source), str(destination))
    moved_pairs.append((source, destination))
    print("已暂时移走：", source.name)

print("\n现在可以 Quick Save 并保存 output。")

已暂时移走： train_multi_inputs_matrix.npz
已暂时移走： test_multi_inputs_matrix.npz
已暂时移走： train_multi_targets_matrix.npz

现在可以 Quick Save 并保存 output。


In [ ]:
!du -sh /kaggle/working

In [ ]:
from pathlib import Path
import shutil

temporary = Path("/kaggle/working/multiome_sparse")
original = Path("/kaggle/working/multiome_sparse")

names = [
    "train_multi_inputs_matrix.npz",
    "test_multi_inputs_matrix.npz",
    "train_multi_targets_matrix.npz",
]

# 全部检查通过后再移动，避免覆盖文件
for name in names:
    if not (temporary / name).is_file():
        raise FileNotFoundError(temporary / name)
    if (original / name).exists():
        raise FileExistsError(original / name)

for name in names:
    shutil.move(str(temporary / name), str(original / name))

print("三个矩阵已恢复到原来的位置。")

In [135]:
from pathlib import Path
import shutil

# moved_pairs 是第一段代码保存的“原路径—临时路径”对应关系
if "moved_pairs" not in globals():
    raise RuntimeError("当前 Session 中找不到 moved_pairs，请使用下面的备用恢复代码。")

# 先统一检查，防止恢复到一半才报错
for original_path, temporary_path in moved_pairs:
    if not temporary_path.is_file():
        raise FileNotFoundError(f"临时文件不存在：{temporary_path}")

    if original_path.exists():
        raise FileExistsError(f"原位置已经存在同名文件：{original_path}")

# 全部检查通过后恢复
for original_path, temporary_path in moved_pairs:
    original_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(temporary_path), str(original_path))
    print("已恢复：", original_path)

print("\n所有暂时移走的文件均已恢复。")

已恢复： /kaggle/working/multiome_sparse/train_multi_inputs_matrix.npz
已恢复： /kaggle/working/multiome_sparse/test_multi_inputs_matrix.npz
已恢复： /kaggle/working/multiome_sparse/train_multi_targets_matrix.npz

所有暂时移走的文件均已恢复。


In [ ]:
from pathlib import Path
import shutil

# moved_pairs 是第一段代码保存的“原路径—临时路径”对应关系
if "moved_pairs" not in globals():
    raise RuntimeError("当前 Session 中找不到 moved_pairs，请使用下面的备用恢复代码。")

# 先统一检查，防止恢复到一半才报错
for original_path, temporary_path in moved_pairs:
    if not temporary_path.is_file():
        raise FileNotFoundError(f"临时文件不存在：{temporary_path}")

    if original_path.exists():
        raise FileExistsError(f"原位置已经存在同名文件：{original_path}")

# 全部检查通过后恢复
for original_path, temporary_path in moved_pairs:
    original_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(temporary_path), str(original_path))
    print("已恢复：", original_path)

print("\n所有暂时移走的文件均已恢复。")

In [ ]:
%cd /kaggle/working

from pathlib import Path
from IPython.display import display, FileLink

names = [
    "train_multi_targets_matrix.npz",  # 先试较小的 824 MB 文件
    "test_multi_inputs_matrix.npz",
    "train_multi_inputs_matrix.npz",
]

for name in names:
    path = Path("multiome_sparse") / name
    if path.is_file():
        display(FileLink(str(path)))
    else:
        print(f"文件不在原位置：{path}")

In [132]:
import os
import zipfile

WORKING_DIR = "/kaggle/working"
ZIP_PATH = "/kaggle/working/multiome_final_backup.zip"

# 避免把之前生成的压缩包再次压进去
EXCLUDE_FILES = {
    os.path.basename(ZIP_PATH),
}

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as zf:
    for root, dirs, files in os.walk(WORKING_DIR):
        for filename in files:
            if filename in EXCLUDE_FILES:
                continue

            file_path = os.path.join(root, filename)
            archive_name = os.path.relpath(file_path, WORKING_DIR)
            zf.write(file_path, archive_name)

print("备份文件已生成：", ZIP_PATH)
print(f"ZIP 大小：{os.path.getsize(ZIP_PATH) / 1024**2:.2f} MB")

备份文件已生成： /kaggle/working/multiome_final_backup.zip
ZIP 大小：886.13 MB


In [134]:
import os
from pathlib import Path
from IPython.display import FileLink, display

os.chdir("/kaggle/working")

zip_name = "multiome_final_backup.zip"
zip_path = Path(zip_name)

if not zip_path.is_file():
    raise FileNotFoundError(f"找不到文件：{zip_path.resolve()}")

print(f"文件大小：{zip_path.stat().st_size / 1024**2:.2f} MB")

display(FileLink(zip_name))

文件大小：886.13 MB


/kaggle/working/multiome_final_backup.zip

单独保存模型本身的备份

In [136]:
import os
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

ROOT = Path("/kaggle/working")
ZIP_NAME = "multiome_models_backup.zip"
ZIP_PATH = ROOT / ZIP_NAME

model_files = []

# 模型、SVD和scaler
for pattern in ("*.keras", "*.joblib"):
    model_files.extend(ROOT.rglob(pattern))

# 输出基因顺序也是模型解释所必需的
for metadata_name in [
    "target_gene_ids.npy",
]:
    model_files.extend(ROOT.rglob(metadata_name))

# 去重并排除ZIP自身
model_files = sorted({
    path.resolve()
    for path in model_files
    if path.is_file() and path.resolve() != ZIP_PATH.resolve()
})

if not model_files:
    raise FileNotFoundError("没有找到 .keras 或 .joblib 模型文件。")

print("准备备份以下模型文件：")
for path in model_files:
    print(f"  {path.relative_to(ROOT)}  "
          f"({path.stat().st_size / 1024**2:.2f} MB)")

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as zf:

    manifest_lines = []

    for path in model_files:
        relative_path = path.relative_to(ROOT)
        zf.write(path, arcname=str(relative_path))
        manifest_lines.append(str(relative_path))

    # 在ZIP内部附带文件清单
    zf.writestr(
        "MODEL_MANIFEST.txt",
        "\n".join(manifest_lines)
    )

print("\n模型备份已生成：", ZIP_PATH)
print(f"ZIP大小：{ZIP_PATH.stat().st_size / 1024**2:.2f} MB")

准备备份以下模型文件：
  multiome_final/final_mlp_seed_123.keras  (1.91 MB)
  multiome_final/final_mlp_seed_2026.keras  (1.91 MB)
  multiome_final/final_mlp_seed_2027.keras  (1.91 MB)
  multiome_final/final_mlp_seed_42.keras  (1.91 MB)
  multiome_final/final_mlp_seed_7.keras  (1.91 MB)
  multiome_final/target_gene_ids.npy  (1.34 MB)
  multiome_final/target_svd256_final.joblib  (22.87 MB)
  multiome_final/x_scaler_final.joblib  (0.00 MB)
  multiome_final/z_scaler_final.joblib  (0.01 MB)
  multiome_sparse/input_multi_svd256.joblib  (223.58 MB)
  multiome_sparse/validation_mlp_x128_y256.keras  (3.04 MB)
  multiome_sparse/validation_mlp_x_scaler.joblib  (0.00 MB)
  multiome_sparse/validation_mlp_z_scaler.joblib  (0.01 MB)
  multiome_sparse/validation_target_svd256.joblib  (22.87 MB)

模型备份已生成： /kaggle/working/multiome_models_backup.zip
ZIP大小：262.28 MB


In [137]:
import os
from IPython.display import FileLink, display

os.chdir("/kaggle/working")
display(FileLink("multiome_models_backup.zip"))

/kaggle/working/multiome_models_backup.zip